# Interview Questions: Python Data Analysis

This notebook collects high-frequency interview topics from NumPy and pandas.

## Part 1: NumPy Deep Copy vs Shallow Copy

**Interview focus:** when two arrays share the same underlying data, modifying one may unexpectedly change the other.

## 1.1 The Key Idea

In NumPy, the most important question is:

> Does this new array share memory with the original array?

If two arrays share memory, changing values through one array can change the other.

| Operation | Result | Shares data? | Interview note |
|---|---|---:|---|
| `b = a` | alias / reference | Yes | Same object, not a real copy |
| `a[1:4]` | view | Usually yes | Slicing commonly returns a view |
| `a.view()` | view | Yes | New array object, same data buffer |
| `a.copy()` | copy | No | Independent array data |
| `a[[1, 3]]` | copy | No | Fancy indexing returns a copy |
| `a[a > 0]` | copy | No | Boolean indexing returns a copy |

In [49]:
import numpy as np
import copy

## 1.2 Assignment: Alias, Not a Copy

`b = a` does not create a new array. It makes `b` point to the same array object as `a`.

面试里要说清楚：assignment is a reference binding, not shallow copy.

### 1.2.1 For Numpy

#### 1.2.1.1 Pure Numerics

In [19]:
a = np.array([1, 2, 3, 4])
b = a

b[0] = 99

print("a:", a)
print("b:", b)
print("a is b:", a is b)
print("shares memory:", np.shares_memory(a, b))

a: [99  2  3  4]
b: [99  2  3  4]
a is b: True
shares memory: True


In [20]:
a_2d = np.array([[1, 2], [3, 4]])
b_2d = a_2d
b_2d[:,0] = 0
print("a_2d:\n", a_2d)
print("b_2d:\n", b_2d)
print("a_2d is b_2d:", a_2d is b_2d)
print("shares memory:", np.shares_memory(a_2d, b_2d))

a_2d:
 [[0 2]
 [0 4]]
b_2d:
 [[0 2]
 [0 4]]
a_2d is b_2d: True
shares memory: True


In [21]:
b_2d[1][1] = 88
print("a_2d:\n", a_2d)
print("b_2d:\n", b_2d)

a_2d:
 [[ 0  2]
 [ 0 88]]
b_2d:
 [[ 0  2]
 [ 0 88]]


#### 1.2.1.2 Array of Objects

In [22]:
a_obj = np.array([{'foo':1,'boo':2}], dtype=object)
b_obj = a_obj
b_obj[0]['foo'] = 99
print("a_obj:", a_obj)
print("b_obj:", b_obj)
print("a_obj is b_obj:", a_obj is b_obj)
print("shares memory:", np.shares_memory(a_obj, b_obj))

a_obj: [{'foo': 99, 'boo': 2}]
b_obj: [{'foo': 99, 'boo': 2}]
a_obj is b_obj: True
shares memory: True


### 1.2.2 For Common Containers

#### 1.2.2.1 Pure Numerics

In [28]:
a_list = [12,312,123]
b_list = a_list
b_list[0] = 99
print("a_list:", a_list)
print("b_list:", b_list)
print("a_list is b_list:", a_list is b_list)
id(a_list) == id(b_list)

a_list: [99, 312, 123]
b_list: [99, 312, 123]
a_list is b_list: True


True

In [30]:
a_nested_list = [[1,2],[3,4]]
b_nested_list = a_nested_list
b_nested_list[0][0] = 99
print("a_nested_list:", a_nested_list)
print("b_nested_list:", b_nested_list)
print("a_nested_list is b_nested_list:", a_nested_list is b_nested_list)    
print(id(a_nested_list) == id(b_nested_list))

a_nested_list: [[99, 2], [3, 4]]
b_nested_list: [[99, 2], [3, 4]]
a_nested_list is b_nested_list: True
True


In [31]:
b_nested_list[0] = [88,99]
print("a_nested_list:", a_nested_list)
print("b_nested_list:", b_nested_list)  

a_nested_list: [[88, 99], [3, 4]]
b_nested_list: [[88, 99], [3, 4]]


#### 1.2.2.2 Container of Nested Objects

In [33]:
a_list_obj = [{'foo':1,'boo':2}, {'car':3,'truck':4}]
b_list_obj = a_list_obj
b_list_obj[0] = 99
print("a_list_obj:", a_list_obj)
print("b_list_obj:", b_list_obj)
print("a_list_obj is b_list_obj:", a_list_obj is b_list_obj)
print(id(a_list_obj) == id(b_list_obj))

a_list_obj: [99, {'car': 3, 'truck': 4}]
b_list_obj: [99, {'car': 3, 'truck': 4}]
a_list_obj is b_list_obj: True
True


In [34]:
b_list_obj[1]['new'] = 'dict'
print("a_list_obj:", a_list_obj)    
print("b_list_obj:", b_list_obj)

a_list_obj: [99, {'car': 3, 'truck': 4, 'new': 'dict'}]
b_list_obj: [99, {'car': 3, 'truck': 4, 'new': 'dict'}]


**Takeaway:** if the interviewer asks "is `b = a` a shallow copy?", the precise answer is **no**. It is the same object reference.

## 1.3 Slicing: Usually a View / Shallow Behavior

A slice often creates a **view**: a new array object that looks at the same underlying data.

This is why modifying the slice can modify the original array.

### 1.3.1 For Numpy

#### 1.3.1.1 Pure Numerics

In [35]:
a = np.array([10, 20, 30, 40, 50])
view_slice = a[1:4]

view_slice[0] = 999

print("a:", a)
print("view_slice:", view_slice)
print("a is view_slice:", a is view_slice)
print("shares memory:", np.shares_memory(a, view_slice))

a: [ 10 999  30  40  50]
view_slice: [999  30  40]
a is view_slice: False
shares memory: True


In [44]:
a_2d = np.array([[1, 2], [3, 4]])
b_2d = a_2d[:]
b_2d[:,0] = 0
print("a_2d:\n", a_2d)
print("b_2d:\n", b_2d)
print("a_2d is b_2d:", a_2d is b_2d)
print("shares memory:", np.shares_memory(a_2d, b_2d))

a_2d:
 [[0 2]
 [0 4]]
b_2d:
 [[0 2]
 [0 4]]
a_2d is b_2d: False
shares memory: True


In [42]:
print(a_2d[:1,1:])
print(a_2d[:1,1:].shape)

print(a_2d[:1,:])
print(a_2d[:1,:].shape)

print(a_2d[:1])
print(a_2d[:1].shape)

[[2]]
(1, 1)
[[1 2]]
(1, 2)
[[1 2]]
(1, 2)


 Caveat!!!

In [45]:
b_2d[1] = [88,99]
print("a_2d:\n", a_2d)
print("b_2d:\n", b_2d)
# print("a_2d is b_2d:", a_2d is b_2d)
# print("shares memory:", np.shares_memory(a_2d, b_2d))

a_2d:
 [[ 0  2]
 [88 99]]
b_2d:
 [[ 0  2]
 [88 99]]


#### 1.3.1.2 Array of Objects

In [46]:
a_arr_obj = np.array([{'foo':1,'boo':2}, {'car':3,'truck':4}], dtype=object)
b_arr_obj = a_arr_obj[:]
b_arr_obj[0] = 99
print("a_arr_obj:", a_arr_obj)
print("b_arr_obj:", b_arr_obj)
print("a_arr_obj is b_arr_obj:", a_arr_obj is b_arr_obj)
print("shares memory:", np.shares_memory(a_arr_obj, b_arr_obj))

a_arr_obj: [99 {'car': 3, 'truck': 4}]
b_arr_obj: [99 {'car': 3, 'truck': 4}]
a_arr_obj is b_arr_obj: False
shares memory: True


In [47]:
b_arr_obj[1]['new'] = 'dict'
print("a_arr_obj:", a_arr_obj)
print("b_arr_obj:", b_arr_obj)

a_arr_obj: [99 {'car': 3, 'truck': 4, 'new': 'dict'}]
b_arr_obj: [99 {'car': 3, 'truck': 4, 'new': 'dict'}]


### 1.3.2 For Common Containers

#### 1.3.2.1 Pure Numerics

In [51]:
a_list_slice = [10, 20, 30, 40, 50]
view_slice = a_list_slice[1:4]  
view_slice[0] = 999
print("a_list_slice:", a_list_slice)
print("view_slice:", view_slice)
print("a_list_slice is view_slice:", a_list_slice is view_slice)    
print(id(a_list_slice) == id(view_slice))

a_list_slice: [10, 20, 30, 40, 50]
view_slice: [999, 30, 40]
a_list_slice is view_slice: False
False


In [50]:
shallow_copy_list = copy.copy(a_list_slice)
shallow_copy_list[0] = 888
print("a_list_slice:", a_list_slice)
print("shallow_copy_list:", shallow_copy_list)
print("a_list_slice is shallow_copy_list:", a_list_slice is shallow_copy_list)
print(id(a_list_slice) == id(shallow_copy_list))

a_list_slice: [10, 20, 30, 40, 50]
shallow_copy_list: [888, 20, 30, 40, 50]
a_list_slice is shallow_copy_list: False
False


In [54]:
a_list_nested_slice = [[10, 20], [30, 40], [50, 60]]
shallow_copy_nested_slice = copy.copy(a_list_nested_slice)
shallow_copy_nested_slice[0][0] = 999
print("a_list_nested_slice:", a_list_nested_slice)
print("shallow_copy_nested_slice:", shallow_copy_nested_slice)
print("a_list_nested_slice is shallow_copy_nested_slice:", a_list_nested_slice is shallow_copy_nested_slice)
print(id(a_list_nested_slice) == id(shallow_copy_nested_slice))

a_list_nested_slice: [[999, 20], [30, 40], [50, 60]]
shallow_copy_nested_slice: [[999, 20], [30, 40], [50, 60]]
a_list_nested_slice is shallow_copy_nested_slice: False
False


a_list_nested_slice ----------> outer list A
                                 [ ref to inner1, ref to inner2, ref to inner3 ]

shallow_copy_nested_slice ----> outer list B
                                 [ ref to inner1, ref to inner2, ref to inner3 ]

inner1 -----------------------> [10, 20]

In [56]:
shallow_copy_nested_slice[0] = [999, 30]
print("a_list_nested_slice:", a_list_nested_slice)
print("shallow_copy_nested_slice:", shallow_copy_nested_slice)

# Because now you are replacing the reference in the copied outer list, not mutating the shared inner list.

a_list_nested_slice: [[999, 20], [30, 40], [50, 60]]
shallow_copy_nested_slice: [[999, 30], [30, 40], [50, 60]]


**Interview intuition:** NumPy prefers views for slicing because it is fast and memory-efficient. But this can cause bugs when you expect independent data.

## 1.4 `view()`: New Array Object, Same Data

`view()` creates a new NumPy array object, but it still shares the same data buffer.

Changing values through the view changes the original.

### 1.4.1 For Numpy

#### 1.4.1.1 Pure Numerics

In [4]:
a = np.array([1, 2, 3, 4])
v = a.view()

v[1] = 200

print("a:", a)
print("v:", v)
print("a is v:", a is v)
print("shares memory:", np.shares_memory(a, v))

a: [  1 200   3   4]
v: [  1 200   3   4]
a is v: False
shares memory: True


In [59]:
arr_2d = np.array([[1, 2], [3, 4]])
arr_2d_view = arr_2d.view()
arr_2d_view[:,0] = 0
print("arr_2d:\n", arr_2d)
print("arr_2d_view:\n", arr_2d_view)
print("arr_2d is arr_2d_view:", arr_2d is arr_2d_view)
print("shares memory:", np.shares_memory(arr_2d, arr_2d_view))

arr_2d:
 [[0 2]
 [0 4]]
arr_2d_view:
 [[0 2]
 [0 4]]
arr_2d is arr_2d_view: False
shares memory: True


#### 1.4.1.2 Array of Objects

In [60]:
arr_obj = np.array([{'foo':1,'boo':2}, {'car':3,'truck':4}], dtype=object)
arr_obj_view = arr_obj.view()
arr_obj_view[0] = 99
print("arr_obj:", arr_obj)
print("arr_obj_view:", arr_obj_view)
print("arr_obj is arr_obj_view:", arr_obj is arr_obj_view)
print("shares memory:", np.shares_memory(arr_obj, arr_obj_view))

arr_obj: [99 {'car': 3, 'truck': 4}]
arr_obj_view: [99 {'car': 3, 'truck': 4}]
arr_obj is arr_obj_view: False
shares memory: True


**Useful distinction:** `a is v` is `False`, because they are different array objects. But `np.shares_memory(a, v)` is `True`, because their data is shared.

## 1.5 `copy()`: Independent Data

`copy()` creates a new array with its own data buffer.

Changing the copy will not change the original numeric array.

In [61]:
a = np.array([10, 20, 30, 40])
c = a.copy()

c[0] = -1

print("a:", a)
print("c:", c)
print("a is c:", a is c)
print("shares memory:", np.shares_memory(a, c))

a: [10 20 30 40]
c: [-1 20 30 40]
a is c: False
shares memory: False


**Practical rule:** when you plan to mutate a slice/result and you do not want to affect the original, call `.copy()` explicitly.

## 1.6 Fancy Indexing and Boolean Indexing Return Copies

This is a very common interview trap:

- Slicing with `:` usually returns a view.
- Fancy indexing with a list/array usually returns a copy.
- Boolean indexing usually returns a copy.

In [62]:
a = np.array([10, 20, 30, 40, 50])

slice_result = a[1:4]
fancy_result = a[[1, 2, 3]]
boolean_result = a[a > 20]

print("slice shares memory:", np.shares_memory(a, slice_result))
print("fancy shares memory:", np.shares_memory(a, fancy_result))
print("boolean shares memory:", np.shares_memory(a, boolean_result))

slice shares memory: True
fancy shares memory: False
boolean shares memory: False


In [63]:
a = np.array([10, 20, 30, 40, 50])

slice_result = a[1:4]
fancy_result = a[[1, 2, 3]]

slice_result[0] = 999
fancy_result[0] = -1

print("a after modifying slice_result and fancy_result:", a)
print("slice_result:", slice_result)
print("fancy_result:", fancy_result)

a after modifying slice_result and fancy_result: [ 10 999  30  40  50]
slice_result: [999  30  40]
fancy_result: [-1 30 40]


**Interview explanation:** `fancy_result[0] = -1` does not modify `a`, because fancy indexing already produced a separate copied array.

## 1.7 Advanced Caveat: Object Arrays

For normal numeric arrays, `.copy()` gives independent numeric data.

But if the array stores Python objects, `.copy()` copies the array container, not necessarily the inner Python objects.

This is less common in data-analysis NumPy code, but it is useful interview knowledge.

In [18]:
records = np.array([{"score": 10}, {"score": 20}], dtype=object)
records_copy = records.copy()

records_copy[0]["score"] = 999

print("records:", records)
print("records_copy:", records_copy)
print("shares memory:", np.shares_memory(records, records_copy))

records: [{'score': 999} {'score': 20}]
records_copy: [{'score': 999} {'score': 20}]
shares memory: False


**Why this happens:** the NumPy array data buffer is copied, but both arrays still contain references to the same inner Python dictionary objects.

For deeply nested Python objects, use `copy.deepcopy`, but in most NumPy/pandas interview problems, prefer numeric arrays and avoid object arrays unless needed.

### 1.7.2 Deepcopy

In [64]:
records = np.array([{"score": 10}, {"score": 20}], dtype=object)
records_deepcopy = copy.deepcopy(records)

records_deepcopy[0]["score"] = 999

print("records:", records)
print("records_deepcopy:", records_deepcopy)
print("shares memory:", np.shares_memory(records, records_deepcopy))

records: [{'score': 10} {'score': 20}]
records_deepcopy: [{'score': 999} {'score': 20}]
shares memory: False


## 1.8 Common Interview Questions

1. What is the difference between `b = a`, `a.view()`, and `a.copy()`?
2. Why does modifying `a[1:3]` sometimes modify the original array?
3. Does fancy indexing return a view or a copy?
4. How can you check whether two arrays share memory?
5. When should you call `.copy()` explicitly in real data work?

Short answer template:

> In NumPy, assignment creates another reference to the same array object. Slicing often returns a view that shares memory with the original. `copy()` creates independent array data. Fancy indexing and boolean indexing usually return copies. I can check memory sharing with `np.shares_memory`.

## 1.9 Common Pitfalls

- Saying `b = a` is a shallow copy. It is more precise to call it an alias/reference.
- Assuming every subset operation returns a copy. Slices often return views.
- Assuming every subset operation returns a view. Fancy indexing and boolean indexing usually return copies.
- Forgetting to call `.copy()` before mutating a slice that should be independent.
- Using object arrays and expecting `.copy()` to recursively copy inner Python objects.

## 1.10 Mini Exercises

Predict the output before running each cell.

In [65]:
# Exercise 1: Will x change?
x = np.array([1, 2, 3, 4, 5])
y = x[::2]
y[1] = 100

print("x:", x)
print("y:", y)
print("shares memory:", np.shares_memory(x, y))

x: [  1   2 100   4   5]
y: [  1 100   5]
shares memory: True


In [66]:
# Exercise 2: Will x change?
x = np.array([1, 2, 3, 4, 5])
y = x[[0, 2, 4]]
y[1] = 100

print("x:", x)
print("y:", y)
print("shares memory:", np.shares_memory(x, y))

x: [1 2 3 4 5]
y: [  1 100   5]
shares memory: False


In [68]:
# Exercise 3: Make this safe by adding copy() in the right place.
scores = np.array([70, 80, 90, 100])
top_scores = scores[2:].copy()
top_scores -= 10

print("scores:", scores)
print("top_scores:", top_scores)

scores: [ 70  80  90 100]
top_scores: [80 90]


## Part 2: pandas Basics Interview Points

This section summarizes interview-relevant pandas points from Section 2 of `da_learn.ipynb`.

**Main interview theme:** pandas is built around labeled data. Most surprising behavior comes from index labels, alignment, missing values, and choosing the right selection method.

## 2.1 Series: Like an Ordered Dictionary with an Index

A `Series` is a 1D labeled array. Interviewers often check whether you understand that the index is not just decoration; it controls selection and alignment.

Key points:

- A `Series` has values plus an index.
- It can be created from a list, NumPy array, or dictionary.
- Dictionary keys become index labels.
- Missing labels become `NaN` when you reindex or request labels that do not exist.
- Use `.loc` for labels and `.iloc` for positions.

In [69]:
import pandas as pd

scores = pd.Series([88, 92, 75], index=["Alice", "Bob", "Charlie"])
print(scores)
print("Label selection:", scores.loc["Bob"])
print("Position selection:", scores.iloc[1])

Alice      88
Bob        92
Charlie    75
dtype: int64
Label selection: 92
Position selection: 92


In [70]:
salary_by_state = {"Ohio": 35000, "Texas": 71000, "Oregon": 16000}
states = ["California", "Ohio", "Oregon", "Texas"]

salaries = pd.Series(salary_by_state, index=states)
print(salaries)
print("Missing values:")
print(salaries[salaries.isna()])

California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
dtype: float64
Missing values:
California   NaN
dtype: float64


**Interview answer template:**

> A pandas `Series` is like a labeled NumPy array or ordered dictionary. The index labels matter because selection, reindexing, and arithmetic alignment are based on labels, not only positions.

## 2.2 DataFrame: Columns Can Have Different Types

A `DataFrame` is a 2D table with row labels and column labels.

Key points:

- Each column is like a `Series`.
- Different columns can have different dtypes.
- Selecting one column usually returns a `Series`.
- Selecting multiple columns returns a `DataFrame`.
- `df.to_numpy()` gives the underlying values, but you lose index/column labels.

In [71]:
data = {
    "state": ["Ohio", "Ohio", "Nevada"],
    "year": [2000, 2001, 2001],
    "population": [1.5, 1.7, 2.4],
}

frame = pd.DataFrame(data)
print(frame)
print("\nSingle column type:", type(frame["state"]).__name__)
print("Multiple columns type:", type(frame[["state", "population"]]).__name__)

    state  year  population
0    Ohio  2000         1.5
1    Ohio  2001         1.7
2  Nevada  2001         2.4

Single column type: Series
Multiple columns type: DataFrame


## 2.3 `.loc` vs `.iloc`: High-Frequency Interview Topic

This is one of the highest-yield pandas interview topics.

| Method | Meaning | Example |
|---|---|---|
| `.loc` | label-based selection | `df.loc["Ohio", "population"]` |
| `.iloc` | integer-position-based selection | `df.iloc[0, 2]` |

Common trap: integer-looking indexes can make `series[0]` ambiguous or confusing. In interviews and production code, prefer explicit `.loc` or `.iloc`.

In [72]:
df = pd.DataFrame(
    np.arange(16).reshape(4, 4),
    index=["Ohio", "Colorado", "Utah", "New York"],
    columns=["one", "two", "three", "four"],
)

print(df)
print("\n.loc by labels:")
print(df.loc[["Colorado", "Utah"], ["two", "four"]])

print("\n.iloc by positions:")
print(df.iloc[[1, 2], [1, 3]])

          one  two  three  four
Ohio        0    1      2     3
Colorado    4    5      6     7
Utah        8    9     10    11
New York   12   13     14    15

.loc by labels:
          two  four
Colorado    5     7
Utah        9    11

.iloc by positions:
          two  four
Colorado    5     7
Utah        9    11


In [73]:
# Boolean filtering: very common data analyst / DS interview operation
high_three = df.loc[df["three"] > 5, ["one", "three"]]
print(high_three)

          one  three
Colorado    4      6
Utah        8     10
New York   12     14


**Common mistake:**

```python
df.iloc[:, :3][df["three"] > 5]
```

This may work, but chained selection is harder to reason about. Prefer one clear `.loc` expression:

```python
df.loc[df["three"] > 5, ["one", "two", "three"]]
```

## 2.4 Reindexing, Dropping, and Missing Values

`reindex` changes the labels to match a requested index/columns. If a requested label does not exist, pandas creates missing values.

Interview-relevant points:

- `reindex` is useful for aligning data to a desired order or schema.
- Missing new labels become `NaN` unless filled.
- `drop` removes labels from rows or columns.
- Use `index=` and `columns=` explicitly for readability.

In [74]:
s = pd.Series([4.5, 7.2, -5.3], index=["d", "b", "a"])
print("Original:")
print(s)

print("\nReindexed:")
print(s.reindex(["a", "b", "c", "d"]))

Original:
d    4.5
b    7.2
a   -5.3
dtype: float64

Reindexed:
a   -5.3
b    7.2
c    NaN
d    4.5
dtype: float64


In [ ]:
cleaned = df.drop(index=["Colorado"], columns=["four"])
print(cleaned)

## 2.5 Alignment: pandas Arithmetic Uses Labels

This is a classic pandas interview concept.

When adding two `Series` or `DataFrame` objects, pandas aligns by labels. If a label exists on one side but not the other, the result becomes `NaN` for that label.

This is powerful for real data because rows may arrive in different orders. It is also a source of bugs if you expected raw positional addition.

In [75]:
s1 = pd.Series([10, 20, 30], index=["a", "b", "c"])
s2 = pd.Series([1, 2, 3], index=["b", "c", "d"])

print(s1 + s2)

a     NaN
b    21.0
c    32.0
d     NaN
dtype: float64


In [ ]:
left = pd.DataFrame(
    np.arange(6).reshape(2, 3),
    index=["row1", "row2"],
    columns=["a", "b", "c"],
)
right = pd.DataFrame(
    np.ones((2, 3)),
    index=["row2", "row3"],
    columns=["b", "c", "d"],
)

print(left + right)
print("\nUse fill_value when missing should behave like 0:")
print(left.add(right, fill_value=0))

**Interview answer template:**

> pandas arithmetic aligns on index and column labels. Non-overlapping labels produce missing values. If I want missing values to behave like zero, I can use arithmetic methods such as `.add(..., fill_value=0)`.

## 2.6 DataFrame and Series Broadcasting

When subtracting a `Series` from a `DataFrame`, pandas aligns the `Series` index with the `DataFrame` columns by default.

This is similar to NumPy broadcasting, but label-aware.

In [76]:
frame = pd.DataFrame(
    np.arange(12).reshape(4, 3),
    columns=list("bde"),
    index=["Utah", "Ohio", "Texas", "Oregon"],
)

first_row = frame.iloc[0]
print("Frame:")
print(frame)
print("\nSubtract first row from every row:")
print(frame - first_row)

Frame:
        b   d   e
Utah    0   1   2
Ohio    3   4   5
Texas   6   7   8
Oregon  9  10  11

Subtract first row from every row:
        b  d  e
Utah    0  0  0
Ohio    3  3  3
Texas   6  6  6
Oregon  9  9  9


In [ ]:
# To broadcast down rows instead, specify axis="index".
state_baseline = frame["d"]
print(frame.sub(state_baseline, axis="index"))

## 2.7 `apply`, `map`, and Vectorization

Interviewers may ask when to use `apply`.

Practical hierarchy:

1. Prefer built-in vectorized pandas/NumPy operations.
2. Use `.map` for element-wise transformation on a `Series`, or element-wise DataFrame mapping where appropriate.
3. Use `.apply` for row-wise/column-wise logic when vectorization is not simple.
4. Avoid row-wise `.apply(axis=1)` for large data when a vectorized solution exists.

In [77]:
frame = pd.DataFrame(
    np.random.default_rng(0).standard_normal((4, 3)),
    columns=list("bde"),
    index=["Utah", "Ohio", "Texas", "Oregon"],
)

range_by_column = frame.apply(lambda col: col.max() - col.min(), axis=0)
range_by_row = frame.apply(lambda row: row.max() - row.min(), axis=1)

print("Range by column:")
print(range_by_column)
print("\nRange by row:")
print(range_by_row)

Range by column:
b    2.569422
d    1.570355
e    1.344158
dtype: float64

Range by row:
Utah      0.772528
Ohio      0.897264
Texas     2.007735
Oregon    1.306747
dtype: float64


In [78]:
formatted = frame.map(lambda x: f"{x:.2f}")
print(formatted)

            b      d      e
Utah     0.13  -0.13   0.64
Ohio     0.10  -0.54   0.36
Texas    1.30   0.95  -0.70
Oregon  -1.27  -0.62   0.04


**Common mistake:** using `.apply` for something pandas can already do directly.

Prefer:

```python
df["x"] + df["y"]
```

instead of:

```python
df.apply(lambda row: row["x"] + row["y"], axis=1)
```

## 2.8 Sorting and Ranking

Useful interview points:

- `sort_index()` sorts by row or column labels.
- `sort_values()` sorts by actual values.
- `rank()` handles ties with methods like `average`, `first`, `min`, `max`, and `dense`.
- Missing values can be placed first or last using `na_position`.

In [84]:
people = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "team": ["A", "A", "B", "B"],
    "score": [90, np.nan, 85, np.nan],
})

print(people.sort_values(["team", "score"], ascending=[True, False], na_position="last"))
print("\nRanks with ties:")
print(people["score"].rank(method="average", ascending=False))
print("\nRanks by first appearance:")
print(people["score"].rank(method="first", ascending=False))

      name team  score
0    Alice    A   90.0
1      Bob    A    NaN
2  Charlie    B   85.0
3    Diana    B    NaN

Ranks with ties:
0    1.0
1    NaN
2    2.0
3    NaN
Name: score, dtype: float64

Ranks by first appearance:
0    1.0
1    NaN
2    2.0
3    NaN
Name: score, dtype: float64


## 2.9 Duplicate Labels

pandas allows duplicate index labels, but many operations become less predictable or harder to reason about.

Interview-relevant advice:

- Check whether indexes are unique with `.index.is_unique`.
- Duplicate labels can make selection return multiple rows instead of one.
- Some operations such as reindexing may fail or behave differently when labels are duplicated.
- In data cleaning, decide whether duplicate labels are meaningful or should be reset/deduplicated.

In [86]:
s = pd.Series([0, 1, 2, 3, 4], index=["a", "a", "b", "b", "c"])
print(s)
print("\nIs index unique?", s.index.is_unique)
print("\nSelecting label 'a' returns:")
print(s.loc["a"])

a    0
a    1
b    2
b    3
c    4
dtype: int64

Is index unique? False

Selecting label 'a' returns:
a    0
a    1
dtype: int64


## 2.10 Summary Stats, Correlation, and Value Counts

For interviews, know the everyday EDA toolkit:

- `.describe()` for quick distribution summary.
- `.count()` counts non-missing values.
- `.idxmin()` / `.idxmax()` return labels of min/max values.
- `.corr()` measures linear correlation.
- `.cov()` measures covariance.
- `.unique()` returns unique values.
- `.value_counts()` returns frequency counts.

In [88]:
sales = pd.DataFrame({
    "region": ["East", "East", "West", "West", "West"],
    "revenue": [100, 120, 90, 150, np.nan],
    "ad_spend": [10, 13, 9, 20, 11],
})

print(sales.describe())
print("\nNon-missing counts:")
print(sales.count())
print("\nRegion counts:")
print(sales["region"].value_counts())

          revenue   ad_spend
count    4.000000   5.000000
mean   115.000000  12.600000
std     26.457513   4.393177
min     90.000000   9.000000
25%     97.500000  10.000000
50%    110.000000  11.000000
75%    127.500000  13.000000
max    150.000000  20.000000

Non-missing counts:
region      5
revenue     4
ad_spend    5
dtype: int64

Region counts:
region
West    3
East    2
Name: count, dtype: int64


In [89]:
print("Correlation between revenue and ad_spend:")
print(sales[["revenue", "ad_spend"]].corr())

Correlation between revenue and ad_spend:
           revenue  ad_spend
revenue   1.000000  0.989325
ad_spend  0.989325  1.000000


## 2.11 Common pandas Interview Questions: Concise Answers

1. **What is the difference between a `Series` and a `DataFrame`?**  
   A `Series` is a 1D labeled array. A `DataFrame` is a 2D labeled table made of columns, where each column is basically a `Series`.

2. **What is the difference between `.loc` and `.iloc`?**  
   `.loc` selects by index/column **labels**. `.iloc` selects by integer **positions**.

3. **Why can pandas arithmetic produce `NaN` even when both objects contain numbers?**  
   Because pandas aligns by index/column labels. If a label exists in one object but not the other, the result for that label becomes `NaN`.

4. **What is data alignment in pandas?**  
   Data alignment means pandas matches data by labels before operations, instead of blindly matching by position.

5. **When would you use `reindex`?**  
   Use `reindex` when you want to reorder rows/columns or force data to match a target set of labels. Missing new labels will produce `NaN`.

6. **What is the difference between `sort_index` and `sort_values`?**  
   `sort_index` sorts by row or column labels. `sort_values` sorts by the actual data values in one or more columns.

7. **What does `rank(method="first")` do differently from the default ranking method?**  
   The default gives tied values the average rank. `method="first"` breaks ties by the order the values appear in the data.

8. **Why can duplicate index labels be dangerous?**  
   Because selecting one label may return multiple rows, and some operations become ambiguous, harder to debug, or may fail.

9. **When should you avoid `.apply(axis=1)`?**  
   Avoid it when a vectorized pandas/NumPy operation can do the same job. Row-wise `apply` is usually slower and less idiomatic.

10. **How would you quickly inspect missing values and category frequencies in a dataset?**  
    Use `df.info()`, `df.isna().sum()`, and for categorical columns use `df["col"].value_counts(dropna=False)`.

## 2.12 Common Pitfalls

- Using `series[0]` instead of explicit `.iloc[0]` or `.loc[label]`.
- Forgetting that pandas aligns by labels during arithmetic.
- Confusing `sort_index()` with `sort_values()`.
- Using chained indexing when one `.loc` expression is clearer.
- Using `.apply(axis=1)` when vectorized code is available.
- Ignoring duplicate index labels.
- Forgetting that `value_counts()` excludes missing values by default unless configured.
- Calling `.to_numpy()` too early and losing useful labels.

## 2.13 Mini Exercises

Predict the output or write the missing line before running.

In [90]:
# Exercise 1: Why does this produce NaN?
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])

print(a + b)

w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64


In [91]:
# Exercise 2: Rewrite this with one clear .loc expression.
# Goal: select rows where score >= 90 and only keep name/team/score.

people = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "team": ["A", "A", "B", "B"],
    "score": [90, 90, 85, 70],
})

answer = people.loc[people["score"] >= 90, ["name", "team", "score"]]
print(answer)

    name team  score
0  Alice    A     90
1    Bob    A     90


In [92]:
# Exercise 3: What does transform-style thinking mean here?
# Add each person's score difference from their team average.

people = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "team": ["A", "A", "B", "B"],
    "score": [90, 80, 70, 100],
})

people["team_avg"] = people.groupby("team")["score"].transform("mean")
people["diff_from_team_avg"] = people["score"] - people["team_avg"]
print(people)

      name team  score  team_avg  diff_from_team_avg
0    Alice    A     90      85.0                 5.0
1      Bob    A     80      85.0                -5.0
2  Charlie    B     70      85.0               -15.0
3    Diana    B    100      85.0                15.0


In [94]:
people.isna().sum()

name                  0
team                  0
score                 0
team_avg              0
diff_from_team_avg    0
dtype: int64

## Part 3: Python Lambda Functions

A `lambda` function is a small anonymous function, usually used when you need a short function for one expression.

**Interview focus:** know when lambdas are useful, when they make code worse, and how they behave with functions like `map`, `filter`, `sorted`, pandas `.apply`, and closures.

## 3.1 Basic Syntax

General form:

```python
lambda arguments: expression
```

A lambda:

- can take zero or more arguments
- must contain exactly one expression
- automatically returns the expression result
- is best for short throwaway logic

It is equivalent to a simple `def`, but usually less readable for complex logic.

In [95]:
square = lambda x: x ** 2

print(square(5))

25


In [96]:
# Equivalent normal function
def square_def(x):
    return x ** 2

print(square_def(5))

25


**Interview note:** lambdas are not more powerful than `def`. They are just shorter for simple expression-based functions.

## 3.2 Multiple Arguments

A lambda can take multiple arguments, just like a normal function.

In [97]:
add = lambda x, y: x + y
weighted_score = lambda exam, project: 0.7 * exam + 0.3 * project

print(add(3, 4))
print(weighted_score(80, 90))

7
83.0


## 3.3 Common Use Case: `sorted(..., key=...)`

One of the cleanest lambda use cases is passing a short sorting key.

In [98]:
students = [
    {"name": "Alice", "score": 88},
    {"name": "Bob", "score": 95},
    {"name": "Charlie", "score": 82},
]

sorted_students = sorted(students, key=lambda row: row["score"], reverse=True)
print(sorted_students)

[{'name': 'Bob', 'score': 95}, {'name': 'Alice', 'score': 88}, {'name': 'Charlie', 'score': 82}]


In [99]:
pairs = [("A", 3), ("B", 1), ("C", 2)]

# Sort by the second element of each tuple
print(sorted(pairs, key=lambda item: item[1]))

[('B', 1), ('C', 2), ('A', 3)]


**Practical rule:** `lambda` works well when the function is obvious from one line, like `lambda row: row["score"]`.

## 3.4 `map`, `filter`, and List Comprehensions

`map` and `filter` often appear in lambda examples, but in real Python code, list comprehensions are often more readable.

In [100]:
numbers = [1, 2, 3, 4, 5]

squared_with_map = list(map(lambda x: x ** 2, numbers))
squared_with_comprehension = [x ** 2 for x in numbers]

print(squared_with_map)
print(squared_with_comprehension)

[1, 4, 9, 16, 25]
[1, 4, 9, 16, 25]


In [102]:
numbers = [1, 2, 3, 4, 5]

filtered_with_filter = list(filter(lambda x: x % 2 == 0, numbers))
filtered_with_comprehension = [x for x in numbers if x % 2 == 0]

print(filtered_with_filter)
print(filtered_with_comprehension)

[2, 4]
[2, 4]


**Interview answer:** `map` applies a function to every item. `filter` keeps items where the function returns `True`. But list comprehensions are often more Pythonic when they are clearer.

## 3.5 Lambda with pandas

In pandas, lambdas are common with `.apply`, `.map`, `assign`, and `groupby` operations.

Important: use vectorized pandas operations when possible. Use lambda when custom logic is small and clear.

In [ ]:
people = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "score": [88, 95, 72],
})

people["passed"] = people["score"].map(lambda x: x >= 80)
print(people)

      name  score  passed
0    Alice     88    True
1      Bob     95    True
2  Charlie     72   False


In [105]:
people = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "exam": [88, 95, 72],
    "project": [90, 85, 80],
})

people["final_score"] = people.apply(
    lambda row: 0.7 * row["exam"] + 0.3 * row["project"],
    axis=1,
)
print(people)

      name  exam  project  final_score
0    Alice    88       90         88.6
1      Bob    95       85         92.0
2  Charlie    72       80         74.4


**Common pandas warning:** row-wise `apply(axis=1)` is convenient but can be slow. If the same logic can be written with vectorized column operations, prefer that.

In [106]:
# Better vectorized version of the previous example
people["final_score_vectorized"] = 0.7 * people["exam"] + 0.3 * people["project"]
print(people)

      name  exam  project  final_score  final_score_vectorized
0    Alice    88       90         88.6                    88.6
1      Bob    95       85         92.0                    92.0
2  Charlie    72       80         74.4                    74.4


## 3.6 Lambda Limitations

A lambda is limited to a single expression. It cannot contain normal multi-line statements like assignment, `for`, `while`, or `try/except` blocks.

Use `def` when:

- the logic needs multiple steps
- the function needs a clear name
- you need comments or debugging
- the expression becomes hard to read

In [ ]:
# Fine as a lambda
label_score = lambda score: "pass" if score >= 80 else "fail"

print(label_score(90))
print(label_score(60))

The example above is okay because the conditional expression is still short. If the logic grows, switch to `def`.

In [ ]:
def label_score_readable(score):
    if score >= 90:
        return "excellent"
    if score >= 80:
        return "pass"
    return "fail"

print(label_score_readable(95))
print(label_score_readable(85))
print(label_score_readable(60))

## 3.7 Closure Gotcha: Lambdas in Loops

A common interview trap: lambdas capture variables by reference, not by value at creation time.

This can surprise you when creating functions inside a loop.

In [107]:
funcs = []
for i in range(3):
    funcs.append(lambda: i)

print([f() for f in funcs])

[2, 2, 2]


Why does this print `[2, 2, 2]`?

Each lambda refers to the same variable `i`. After the loop ends, `i` is `2`, so all functions return `2`.

In [108]:
# Fix: bind the current value using a default argument
funcs = []
for i in range(3):
    funcs.append(lambda i=i: i)

print([f() for f in funcs])

[0, 1, 2]


**Interview answer template:**

> Python closures capture variables, not their immediate values. In a loop, all lambdas may refer to the final loop value unless we bind the current value using a default argument like `lambda i=i: i`.

## 3.8 Lambda vs `def`

| Feature | `lambda` | `def` |
|---|---|---|
| Name | anonymous unless assigned | named function |
| Body | single expression | multiple statements allowed |
| Return | implicit | explicit `return` |
| Best for | short callbacks / keys | reusable or complex logic |
| Readability | good only when simple | better for non-trivial logic |

**Practical rule:** if you need to explain the lambda, it probably should be a `def`.

## 3.9 Common Interview Questions with Concise Answers

1. **What is a lambda function in Python?**  
   A lambda is a small anonymous function defined with `lambda arguments: expression`. It returns the expression result automatically.

2. **How is a lambda different from a normal `def` function?**  
   A lambda is limited to one expression and is usually unnamed. A `def` function can contain multiple statements, comments, and explicit `return` logic.

3. **When should you use a lambda?**  
   Use it for short, simple callback functions, such as sorting keys, quick transformations, or small pandas operations.

4. **When should you avoid lambda?**  
   Avoid it when the logic is complex, reused often, hard to read, or needs multiple statements. Use `def` instead.

5. **Can a lambda contain multiple statements?**  
   No. A lambda can only contain one expression, though that expression can include things like a conditional expression.

6. **What is a common lambda use case with `sorted`?**  
   Passing a key function, such as `sorted(rows, key=lambda row: row["score"])`.

7. **What is the difference between `map(lambda...)` and a list comprehension?**  
   Both can transform items, but list comprehensions are often more readable and Pythonic for simple transformations.

8. **Why can lambdas inside loops behave unexpectedly?**  
   They capture the loop variable itself, not its value at each iteration, so all functions may use the final loop value.

9. **How do you fix the lambda-in-loop closure issue?**  
   Bind the current value as a default argument, such as `lambda i=i: i`.

10. **Should you use lambda with pandas `.apply(axis=1)`?**  
    Sometimes for small custom row-wise logic, but prefer vectorized pandas operations when possible because they are usually faster and clearer.

## 3.10 Common Pitfalls

- Writing lambdas that are too complex to read.
- Using `lambda` when a named `def` would be clearer.
- Forgetting that lambdas can only contain one expression.
- Overusing `map` and `filter` when list comprehensions are clearer.
- Using row-wise pandas `.apply(axis=1)` when vectorized operations are available.
- Creating lambdas inside loops without handling the closure variable issue.

## 3.11 Mini Exercises

Predict the output before running each cell.

In [109]:
# Exercise 1: Sort by string length
words = ["data", "machine", "AI", "statistics"]

answer = sorted(words, key=lambda word: len(word))
print(answer)

['AI', 'data', 'machine', 'statistics']


In [110]:
# Exercise 2: Convert scores to pass/fail labels
scores = [95, 72, 88, 60]

labels = list(map(lambda score: "pass" if score >= 80 else "fail", scores))
print(labels)

['pass', 'fail', 'pass', 'fail']


In [111]:
# Exercise 3: Closure check
funcs = [lambda x=x: x * 10 for x in range(4)]

print([f() for f in funcs])

[0, 10, 20, 30]


## Part 4: Data Cleaning and Preparation Interview Points

This section summarizes interview-relevant data cleaning and preparation topics from Section 4 of `da_learn.ipynb`.

**Main interview theme:** good data cleaning is not just applying functions. It means understanding missingness, preserving useful rows, avoiding silent type bugs, and making transformations reproducible.

## 4.1 Missing Data: Detect, Decide, Then Handle

Common missing-value tools:

- `isna()` / `notna()` detect missing values.
- `dropna()` removes missing rows/columns.
- `fillna()` fills missing values.
- `ffill()` / `bfill()` forward-fill or backward-fill values.
- `thresh=` keeps rows with at least a minimum number of non-missing values.

Interview point: do not blindly drop missing rows. First ask whether missingness itself contains information or bias.

In [113]:
raw = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "age": [25, np.nan, 40, np.nan],
    "income": [50000, 62000, np.nan, np.nan],
    "segment": ["A", "B", None, "A"],
})

display(raw)
print("Missing count by column:")
print(raw.isna().sum())

,customer_id,age,income,segment
0,1,25.0,50000.0,A
1,2,NaN,62000.0,B
2,3,40.0,NaN,None
3,4,NaN,NaN,A


Missing count by column:
customer_id    0
age            2
income         2
segment        1
dtype: int64


In [114]:
print("Drop rows with any missing value:")
print(raw.dropna())

print("Keep rows with at least 3 non-missing values:")
print(raw.dropna(thresh=3))

Drop rows with any missing value:
   customer_id   age   income segment
0            1  25.0  50000.0       A
Keep rows with at least 3 non-missing values:
   customer_id   age   income segment
0            1  25.0  50000.0       A
1            2   NaN  62000.0       B


In [115]:
filled = raw.copy()
filled["age"] = filled["age"].fillna(filled["age"].median())
filled["income"] = filled["income"].fillna(filled["income"].mean())
filled["segment"] = filled["segment"].fillna("Unknown")

print(filled)

   customer_id   age   income  segment
0            1  25.0  50000.0        A
1            2  32.5  62000.0        B
2            3  40.0  56000.0  Unknown
3            4  32.5  56000.0        A


**Interview answer template:**

> I first inspect missingness with `isna().sum()` and percentages. Then I decide whether to drop, impute, flag, or investigate based on the column meaning and missingness pattern. I avoid blindly dropping rows because it may bias the dataset.

## 4.2 Duplicates

Useful tools:

- `duplicated()` marks duplicate rows.
- `drop_duplicates()` removes duplicate rows.
- `subset=` checks duplicates using selected columns.
- `keep="first"`, `keep="last"`, or `keep=False` controls which duplicates remain.

Interview point: define the business key before removing duplicates. A duplicated email, transaction ID, or entire row can mean different things.

In [116]:
orders = pd.DataFrame({
    "order_id": [101, 101, 102, 103, 103],
    "customer": ["Alice", "Alice", "Bob", "Charlie", "Charlie"],
    "amount": [50, 50, 20, 30, 35],
})

display(orders)
print("Duplicate entire rows:")
print(orders.duplicated())

print("Duplicate order_id:")
print(orders.duplicated(subset=["order_id"], keep=False))

,order_id,customer,amount
0,101,Alice,50
1,101,Alice,50
2,102,Bob,20
3,103,Charlie,30
4,103,Charlie,35


Duplicate entire rows:
0    False
1     True
2    False
3    False
4    False
dtype: bool
Duplicate order_id:
0     True
1     True
2    False
3     True
4     True
dtype: bool


In [117]:
# Keep the last record for each order_id, assuming later rows are corrections.
deduped_orders = orders.drop_duplicates(subset=["order_id"], keep="last")
print(deduped_orders)

   order_id customer  amount
1       101    Alice      50
2       102      Bob      20
4       103  Charlie      35


## 4.3 Mapping, Replacing, and Type Conversion

Common cleaning tasks:

- Map raw categories to standard categories with `.map()`.
- Replace sentinel values like `-999` with `NaN` using `.replace()`.
- Convert dtypes with `.astype()` or `pd.to_numeric(..., errors="coerce")`.

Interview point: sentinel values are dangerous because they look numeric but actually mean missing or invalid.

In [118]:
foods = pd.DataFrame({
    "food": ["bacon", "pastrami", "nova lox", "honey ham"],
    "ounces": [4, 6, 6, 5],
})

meat_to_animal = {
    "bacon": "pig",
    "pastrami": "cow",
    "honey ham": "pig",
    "nova lox": "salmon",
}

foods["animal"] = foods["food"].map(meat_to_animal)
print(foods)

        food  ounces  animal
0      bacon       4     pig
1   pastrami       6     cow
2   nova lox       6  salmon
3  honey ham       5     pig


In [119]:
measurements = pd.Series([1.0, -999.0, 2.5, -1000.0, 3.2])
cleaned_measurements = measurements.replace([-999.0, -1000.0], np.nan)

print(cleaned_measurements)

0    1.0
1    NaN
2    2.5
3    NaN
4    3.2
dtype: float64


In [120]:
dirty_numbers = pd.Series(["10", "20", "bad", "40"])
clean_numbers = pd.to_numeric(dirty_numbers, errors="coerce")

print(clean_numbers)
print("Missing after conversion:", clean_numbers.isna().sum())

0    10.0
1    20.0
2     NaN
3    40.0
dtype: float64
Missing after conversion: 1


## 4.4 Renaming Columns and Indexes

Renaming is simple but important for clean downstream analysis.

Useful tools:

- `rename(columns={...})`
- `rename(index={...})`
- string methods such as `.str.lower()` and `.str.replace()` on column names

Interview point: standardize column names early so later code is easier to read and less error-prone.

In [121]:
messy = pd.DataFrame({
    "Customer ID": [1, 2],
    "Total Spend ($)": [100.5, 200.0],
})

messy.columns = (
    messy.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("($)", "usd", regex=False)
)

print(messy)

   customer_id  total_spend_usd
0            1            100.5
1            2            200.0


## 4.5 Binning / Discretization

`pd.cut()` converts continuous values into intervals or named categories.

Use cases:

- age groups
- income bands
- risk buckets
- score bands

Interview point: binning improves interpretability, but it loses numeric detail. Be careful with bin boundaries.

In [122]:
ages = [20, 22, 25, 27, 31, 45, 61]
bins = [18, 25, 35, 60, 100]
labels = ["Youth", "YoungAdult", "MiddleAged", "Senior"]

age_group = pd.cut(ages, bins=bins, labels=labels)
print(age_group)
print(pd.Series(age_group).value_counts(sort=False))

['Youth', 'Youth', 'Youth', 'YoungAdult', 'YoungAdult', 'MiddleAged', 'Senior']
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']
Youth         3
YoungAdult    2
MiddleAged    1
Senior        1
Name: count, dtype: int64


## 4.6 Outlier Detection and Capping

Common first-pass tools:

- `describe()` for quick distribution checks.
- z-score or standard-deviation rules for roughly normal data.
- IQR rules for skewed data.
- clipping/capping values when extreme values are valid but too influential.

Interview point: outliers are not automatically errors. Investigate whether they are data-entry errors, rare valid events, or important business signals.

In [124]:
rng = np.random.default_rng(42)
values = pd.Series(np.r_[rng.normal(0, 1, 20), [8, -7]])

display(values)
print(values.describe())
print("Potential outliers using abs(value) > 3:")
print(values[values.abs() > 3])

0     0.304717
1    -1.039984
2     0.750451
3     0.940565
4    -1.951035
5    -1.302180
6     0.127840
7    -0.316243
8    -0.016801
9    -0.853044
10    0.879398
11    0.777792
12    0.066031
13    1.127241
14    0.467509
15   -0.859292
16    0.368751
17   -0.958883
18    0.878450
19   -0.049926
20    8.000000
21   -7.000000
dtype: float64

count    22.000000
mean      0.015516
std       2.463093
min      -7.000000
25%      -0.857730
50%       0.096936
75%       0.770957
max       8.000000
dtype: float64
Potential outliers using abs(value) > 3:
20    8.0
21   -7.0
dtype: float64


In [ ]:
capped_values = values.clip(lower=-3, upper=3)

print("Original max/min:", values.max(), values.min())
print("Capped max/min:", capped_values.max(), capped_values.min())

## 4.7 Random Sampling and Permutation

Useful tools:

- `df.sample(n=...)` or `df.sample(frac=...)` samples rows.
- `random_state=` makes sampling reproducible.
- `np.random.permutation()` creates a random order.

Interview point: always set a seed/random state when the result needs to be reproducible.

In [130]:
# set a seed for reproducibility
seed = 123
df = pd.DataFrame({"x": range(10), "group": list("AAABBBCCDD")})
display(df)
sample = df.sample(n=4, random_state=seed)
shuffled = df.sample(frac=1, random_state=seed)

print("Sample:")
print(sample)
print("Shuffled:")
print(shuffled.head())

,x,group
0,0,A
1,1,A
2,2,A
3,3,B
4,4,B
5,5,B
6,6,C
7,7,C
8,8,D
9,9,D


Sample:
   x group
4  4     B
0  0     A
7  7     C
5  5     B
Shuffled:
   x group
4  4     B
0  0     A
7  7     C
5  5     B
8  8     D


In [127]:
shuffled

,x,group
2,2,A
8,8,D
4,4,B
9,9,D
1,1,A
6,6,C
7,7,C
3,3,B
0,0,A
5,5,B


## 4.8 Dummy Variables / One-Hot Encoding

`pd.get_dummies()` converts categorical values into indicator columns.

Use cases:

- preparing categorical features for ML models
- converting categories into numeric model input
- building simple frequency/indicator features

Interview point: avoid data leakage. For train/test ML workflows, ensure dummy columns are aligned between train and test sets.

In [131]:
df = pd.DataFrame({
    "city": ["NY", "SF", "NY", "LA"],
    "sales": [100, 150, 120, 90],
})

dummies = pd.get_dummies(df["city"], prefix="city", dtype=int)
model_df = df[["sales"]].join(dummies)

print(model_df)

   sales  city_LA  city_NY  city_SF
0    100        0        1        0
1    150        0        0        1
2    120        0        1        0
3     90        1        0        0


## 4.9 String Cleaning

Common string-cleaning operations:

- `.str.strip()` removes leading/trailing spaces.
- `.str.lower()` standardizes case.
- `.str.replace()` replaces patterns.
- `.str.contains()` detects patterns.
- regex helps with inconsistent spacing or formats.

Interview point: dirty strings often create fake categories, such as `"NY"`, `"ny"`, and `" NY "` being treated as different values.

In [132]:
import re

cities = pd.Series([" NY ", "ny", "New York", "SF", "san francisco "])

clean_cities = (
    cities.str.strip()
    .str.lower()
    .replace({"ny": "new york", "sf": "san francisco"})
)

print(clean_cities.value_counts())

new york         3
san francisco    2
Name: count, dtype: int64


In [133]:
text = "foo    bar	 baz  	qux"
print(re.split(r"\s+", text))

['foo', 'bar', 'baz', 'qux']


## 4.10 Categorical Data

pandas `category` dtype stores repeated string-like values more efficiently.

Benefits:

- can reduce memory usage
- can speed up some groupby/value-count operations
- can represent ordered categories

Interview point: use categorical dtype for repeated low-cardinality string columns, but be careful when categories have a meaningful order.

In [134]:
fruits = pd.DataFrame({
    "fruit": ["apple", "orange", "apple", "banana", "apple"],
    "count": [3, 2, 5, 1, 4],
})

fruits["fruit"] = fruits["fruit"].astype("category")

print(fruits.info())
print("Categories:", list(fruits["fruit"].cat.categories))
print("Codes:", fruits["fruit"].cat.codes.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   fruit   5 non-null      category
 1   count   5 non-null      int64   
dtypes: category(1), int64(1)
memory usage: 309.0 bytes
None
Categories: ['apple', 'banana', 'orange']
Codes: [0, 2, 0, 1, 0]


## 4.11 Common Data Cleaning Interview Questions with Concise Answers

1. **How do you inspect missing values in a DataFrame?**  
   Use `df.info()`, `df.isna().sum()`, and missing percentages like `df.isna().mean()`.

2. **When would you use `dropna()` vs `fillna()`?**  
   Use `dropna()` when missing rows/columns are not useful or too incomplete. Use `fillna()` when you can reasonably impute missing values without biasing the analysis too much.

3. **Why is blindly dropping missing data risky?**  
   It can remove too much data or introduce bias if missingness is related to the outcome or data collection process.

4. **How do you remove duplicates?**  
   Use `duplicated()` to inspect and `drop_duplicates()` to remove, usually with a business-defined `subset` key.

5. **What is a sentinel value?**  
   A special value like `-999` or `unknown` that represents missing or invalid data. It should often be replaced with `NaN` before analysis.

6. **What is the difference between `map` and `replace`?**  
   `map` is often used to transform values using a dictionary or function, especially creating a new derived column. `replace` is often used to substitute specific old values with new values.

7. **Why standardize column names?**  
   Clean column names make code easier to write, read, and debug, especially when columns contain spaces, symbols, or inconsistent casing.

8. **When would you use `pd.cut()`?**  
   Use it to convert continuous numeric values into bins, such as age groups or score bands.

9. **How should you handle outliers?**  
   First investigate whether they are errors or valid extreme values. Then decide whether to keep, remove, cap, transform, or flag them.

10. **Why use `pd.get_dummies()`?**  
    It converts categorical variables into numeric indicator columns for modeling or analysis.

11. **Why is string cleaning important?**  
    Inconsistent spaces, casing, and spelling can create fake categories and wrong counts.

12. **When should you use pandas `category` dtype?**  
    Use it for repeated low-cardinality string columns to save memory and sometimes improve performance.

## 4.12 Common Pitfalls

- Dropping missing rows before checking how much data is lost.
- Filling missing values with the mean when the column is highly skewed.
- Removing duplicates without defining the correct business key.
- Treating sentinel values like `-999` as real numeric values.
- Forgetting to convert dirty numeric strings with `pd.to_numeric`.
- Binning continuous values without checking boundary rules.
- Treating every outlier as an error.
- Creating dummy variables separately for train and test data without aligning columns.
- Forgetting to strip/lowercase strings before counting categories.
- Using `category` dtype without thinking about category ordering.

## 4.13 Mini Exercises

Try to answer before running the cells.

In [135]:
# Exercise 1: Clean sentinel values and compute the mean.
scores = pd.Series([80, 90, -999, 75, -999])

clean_scores = scores.replace(-999, np.nan)
print("Mean after cleaning:", clean_scores.mean())

Mean after cleaning: 81.66666666666667


In [139]:
# Exercise 2: Find duplicated users by email.
users = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "email": ["a@test.com", "b@test.com", "a@test.com", "c@test.com"],
})

print(users[users.duplicated(subset=["email"], keep=False)])
print(users[users.duplicated(subset=["email"], keep='last')])
print(users[~users.duplicated(subset=["email"], keep='last')])

   user_id       email
0        1  a@test.com
2        3  a@test.com
   user_id       email
0        1  a@test.com
   user_id       email
1        2  b@test.com
2        3  a@test.com
3        4  c@test.com


In [140]:
# Exercise 3: Standardize messy category labels.
labels = pd.Series([" Gold", "gold", "SILVER ", "silver", "Bronze"])
clean_labels = labels.str.strip().str.lower().str.title()

print(clean_labels.value_counts())

Gold      2
Silver    2
Bronze    1
Name: count, dtype: int64


## Part 5: Data Wrangling Interview Points

This section summarizes and extends Section 5 of `da_learn.ipynb`: joins, combines, hierarchical indexes, reshaping, pivoting, and melting.

**Main interview theme:** data wrangling is about changing the *shape* and *relationships* of data without accidentally losing rows, duplicating rows, or changing meaning.

## 5.1 The Interview Mental Model

Most wrangling questions reduce to three questions:

1. **What is the key?** Which column(s) uniquely identify a row?
2. **What is the grain?** What does one row represent?
3. **What shape do I need?** Long format, wide format, or joined table?

If you can state the key, grain, and target shape clearly, your pandas code becomes much safer.

## 5.2 Merge / Join: SQL-Style Table Combination

`pd.merge` combines two tables using one or more keys.

Common join types:

| Join type | Keeps |
|---|---|
| `inner` | matching keys only |
| `left` | all rows from the left table |
| `right` | all rows from the right table |
| `outer` | all keys from both tables |

Interview habit: always specify `on=`, `left_on=`, or `right_on=` explicitly. Do not rely on overlapping column names unless it is a quick exploration.

In [146]:
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "customer_id": [1, 2, 2, 5],
    "amount": [80, 40, 120, 60],
})

customers = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "customer_name": ["Alice", "Bob", "Charlie"],
    "segment": ["VIP", "Standard", "VIP"],
})

merged = orders.merge(customers, on="customer_id", how="left")
print(merged)

   order_id  customer_id  amount customer_name   segment
0       101            1      80         Alice       VIP
1       102            2      40           Bob  Standard
2       103            2     120           Bob  Standard
3       104            5      60           NaN       NaN


**How to explain this in interviews:**

> I used a left join because orders are the main table. I want to keep every order even if the customer lookup table is incomplete. Missing customer fields after the merge tell me which orders failed to match.

In [147]:
# Find rows that failed to match the lookup table.
unmatched_orders = merged[merged["customer_name"].isna()]
print(unmatched_orders)

   order_id  customer_id  amount customer_name segment
3       104            5      60           NaN     NaN


## 5.3 Merge Safety Checks: `validate` and `indicator`

Two very useful tools for real projects and interviews:

- `validate=` checks the expected relationship, such as `one_to_one`, `one_to_many`, or `many_to_one`.
- `indicator=True` adds a `_merge` column showing whether each row matched both tables or only one side.

These are excellent for showing mature data-cleaning judgment.

In [148]:
checked_merge = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print(checked_merge)
print("\nMerge status counts:")
print(checked_merge["_merge"].value_counts())

   order_id  customer_id  amount customer_name   segment     _merge
0       101            1      80         Alice       VIP       both
1       102            2      40           Bob  Standard       both
2       103            2     120           Bob  Standard       both
3       104            5      60           NaN       NaN  left_only

Merge status counts:
_merge
both          3
left_only     1
right_only    0
Name: count, dtype: int64


**Interview answer template:**

> Before trusting a merge, I check row counts, duplicate keys, unmatched keys, and use `validate` when I know the expected relationship. This catches accidental many-to-many joins early.

## 5.4 Many-to-Many Merge: The Row Explosion Trap

If both tables contain duplicate keys, a merge can create a Cartesian product for each repeated key.

This is one of the most important pandas interview traps.

In [149]:
left = pd.DataFrame({
    "user_id": [1, 1, 2],
    "event": ["login", "purchase", "login"],
})

right = pd.DataFrame({
    "user_id": [1, 1, 2],
    "coupon": ["A", "B", "C"],
})

many_to_many = left.merge(right, on="user_id", how="inner")
print(many_to_many)
print("\nRows before:", len(left), len(right))
print("Rows after:", len(many_to_many))

   user_id     event coupon
0        1     login      A
1        1     login      B
2        1  purchase      A
3        1  purchase      B
4        2     login      C

Rows before: 3 3
Rows after: 5


Why did `user_id == 1` create 4 rows?

The left table has 2 rows for user 1 and the right table has 2 rows for user 1. The merge creates `2 * 2 = 4` combinations.

**Practical fix:** deduplicate, aggregate, or include more join keys before merging.

In [151]:
# Example fix: aggregate coupons before joining.
coupon_count = right.groupby("user_id", as_index=False).agg(num_coupons=("coupon", "nunique"))
safer = left.merge(coupon_count, on="user_id", how="left", validate="many_to_one")
print(safer)

   user_id     event  num_coupons
0        1     login            2
1        1  purchase            2
2        2     login            1


In [150]:
right.groupby("user_id", as_index=False).agg(num_coupons=("coupon", "nunique"))

,user_id,num_coupons
0,1,2
1,2,1


## 5.5 Anti-Join: Find Records in One Table but Not Another

pandas does not have a named `anti_join` method, but you can build one with `indicator=True`.

Common use cases:

- orders without customers
- users without transactions
- products missing from lookup tables
- test rows whose keys were not present in training data

In [152]:
all_users = pd.DataFrame({"user_id": [1, 2, 3, 4, 5]})
active_users = pd.DataFrame({"user_id": [2, 4]})

anti = all_users.merge(active_users, on="user_id", how="left", indicator=True)
inactive_users = anti.loc[anti["_merge"] == "left_only", ["user_id"]]

print(inactive_users)

   user_id
0        1
2        3
4        5


## 5.6 `merge` vs `join` vs `concat`

| Tool | Best for |
|---|---|
| `merge` | SQL-style joins using columns or indexes |
| `join` | convenient index-based joining, especially joining columns from another DataFrame |
| `concat` | stacking objects along rows or columns |

Simple memory trick:

- `merge`: match rows by keys
- `concat`: glue objects together along an axis
- `join`: often index-based convenience wrapper

In [154]:
jan = pd.DataFrame({"month": "Jan", "sales": [100, 120]}, index=["A", "B"])
feb = pd.DataFrame({"month": "Feb", "sales": [90, 140]}, index=["A", "B"])

stacked_rows = pd.concat([jan, feb], axis="index")
print(stacked_rows)

  month  sales
A   Jan    100
B   Jan    120
A   Feb     90
B   Feb    140


In [153]:
pd.DataFrame({"month": "Jan", "sales": [100, 120]}, index=["A", "B"])

,month,sales
A,Jan,100
B,Jan,120


In [155]:
profile = pd.DataFrame({"team": ["Data", "ML"]}, index=["A", "B"])
quota = pd.DataFrame({"quota": [110, 130]}, index=["A", "B"])

joined_columns = profile.join(quota)
print(joined_columns)

   team  quota
A  Data    110
B    ML    130


## 5.7 Concatenation: Rows vs Columns

`pd.concat` stacks objects together.

- `axis="index"` or `axis=0`: stack rows
- `axis="columns"` or `axis=1`: add columns side by side
- `join="inner"`: keep only shared labels on the other axis
- `keys=`: preserve where each piece came from

In [156]:
north = pd.DataFrame({"store": ["N1", "N2"], "sales": [100, 150]})
south = pd.DataFrame({"store": ["S1", "S2"], "sales": [80, 120]})

all_regions = pd.concat([north, south], axis="index", ignore_index=True)
print(all_regions)

  store  sales
0    N1    100
1    N2    150
2    S1     80
3    S2    120


In [157]:
all_regions_with_keys = pd.concat(
    {"north": north, "south": south},
    names=["region", "row"],
)
print(all_regions_with_keys)

           store  sales
region row             
north  0      N1    100
       1      N2    150
south  0      S1     80
       1      S2    120


## 5.8 `combine_first`: Fill Missing Values from Another Object

`combine_first` is useful when one dataset is primary and another is a fallback.

Example: use corrected values when available, otherwise keep original values.

In [158]:
original_prices = pd.Series({"apple": 1.20, "banana": np.nan, "orange": 1.50})
backup_prices = pd.Series({"banana": 0.80, "orange": 1.45, "pear": 2.00})

combined_prices = original_prices.combine_first(backup_prices)
print(combined_prices)

apple     1.2
banana    0.8
orange    1.5
pear      2.0
dtype: float64


## 5.9 Hierarchical Index / MultiIndex

A MultiIndex means the rows or columns have multiple levels of labels.

Common uses:

- grouped summaries with multiple keys
- panel-like data such as `region -> store -> date`
- reshape workflows with `stack` and `unstack`

Interview point: MultiIndex is powerful, but for everyday analysis you often reset it back into columns for clarity.

In [141]:
sales = pd.DataFrame({
    "region": ["East", "East", "West", "West"],
    "store": ["E1", "E2", "W1", "W2"],
    "revenue": [100, 130, 90, 160],
    "orders": [10, 13, 9, 16],
})

multi = sales.set_index(["region", "store"])
print(multi)
print("\nEast stores:")
print(multi.loc["East"])

              revenue  orders
region store                 
East   E1         100      10
       E2         130      13
West   W1          90       9
       W2         160      16

East stores:
       revenue  orders
store                 
E1         100      10
E2         130      13


In [143]:
summary = multi.groupby(level="region").sum()
print(summary)

back_to_columns = multi.reset_index()
print("\nBack to normal columns:")
print(back_to_columns)

        revenue  orders
region                 
East        230      23
West        250      25

Back to normal columns:
  region store  revenue  orders
0   East    E1      100      10
1   East    E2      130      13
2   West    W1       90       9
3   West    W2      160      16


## 5.10 `stack` and `unstack`: Move Between Index Levels and Columns

- `stack()` moves columns into the row index, usually making data longer.
- `unstack()` moves an index level into columns, usually making data wider.

This is common after groupby results with multiple keys.

In [145]:
scores = pd.DataFrame({
    "student": ["Alice", "Alice", "Bob", "Bob"],
    "subject": ["Math", "English", "Math", "English"],
    "score": [90, 85, 78, 88],
})

indexed_scores = scores.set_index(["student", "subject"])["score"]
print(indexed_scores)

wide_scores = indexed_scores.unstack("subject")
print("\nWide scores:")
print(wide_scores)

long_again = wide_scores.stack().reset_index(name="score")
print("\nLong again:")
print(long_again)

student  subject
Alice    Math       90
         English    85
Bob      Math       78
         English    88
Name: score, dtype: int64

Wide scores:
subject  English  Math
student               
Alice         85    90
Bob           88    78

Long again:
  student  subject  score
0   Alice  English     85
1   Alice     Math     90
2     Bob  English     88
3     Bob     Math     78


## 5.11 Pivot vs Pivot Table vs Melt

| Tool | Use when |
|---|---|
| `pivot` | each `index` + `columns` pair is unique |
| `pivot_table` | duplicates may exist and need aggregation |
| `melt` | convert wide format to long/tidy format |

Interview point: if `pivot` fails because of duplicate entries, use `pivot_table` with an aggregation function.

In [164]:
transactions = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-01", "2026-01-02", "2026-01-02"],
    "channel": ["web", "store", "web", "store"],
    "revenue": [100, 80, 120, 90],
})

display(transactions)
wide = transactions.pivot(index="date", columns="channel", values="revenue")
display(wide)

,date,channel,revenue
0,2026-01-01,web,100
1,2026-01-01,store,80
2,2026-01-02,web,120
3,2026-01-02,store,90


channel,store,web
date,,
2026-01-01,80,100
2026-01-02,90,120


In [165]:
# Duplicate date-channel combinations require pivot_table.
dup_transactions = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-01", "2026-01-01"],
    "channel": ["web", "web", "store"],
    "revenue": [100, 50, 80],
})

pivot_summary = dup_transactions.pivot_table(
    index="date",
    columns="channel",
    values="revenue",
    aggfunc="sum",
)
print(pivot_summary)

channel     store  web
date                  
2026-01-01     80  150


In [166]:
wide_metrics = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-02"],
    "web_revenue": [100, 120],
    "store_revenue": [80, 90],
})

long_metrics = wide_metrics.melt(
    id_vars="date",
    var_name="metric",
    value_name="value",
)
print(long_metrics)

         date         metric  value
0  2026-01-01    web_revenue    100
1  2026-01-02    web_revenue    120
2  2026-01-01  store_revenue     80
3  2026-01-02  store_revenue     90


## 5.12 Tidy Data: A Useful Interview Framing

A tidy dataset usually means:

- each variable is a column
- each observation is a row
- each value is a cell

Many pandas wrangling tasks are about moving between human-readable wide tables and machine-friendly long/tidy tables.

In [167]:
survey_wide = pd.DataFrame({
    "person": ["Alice", "Bob"],
    "q1": [5, 3],
    "q2": [4, 4],
    "q3": [5, 2],
})

survey_long = survey_wide.melt(
    id_vars="person",
    var_name="question",
    value_name="rating",
)
print(survey_long)

  person question  rating
0  Alice       q1       5
1    Bob       q1       3
2  Alice       q2       4
3    Bob       q2       4
4  Alice       q3       5
5    Bob       q3       2


## 5.13 Row-Count Debugging Checklist

Use this before and after joins/reshapes:

1. What is one row supposed to represent?
2. Which key(s) should be unique?
3. Are keys duplicated on either side?
4. How many rows before and after?
5. How many unmatched rows?
6. Did missing values increase unexpectedly?
7. Did numeric totals change unexpectedly?

This checklist is often more impressive in interviews than memorizing every function argument.

In [ ]:
def merge_diagnostics(left, right, key):
    print("Left rows:", len(left))
    print("Right rows:", len(right))
    print("Left duplicate keys:", left.duplicated(subset=[key]).sum())
    print("Right duplicate keys:", right.duplicated(subset=[key]).sum())

    result = left.merge(right, on=key, how="left", indicator=True)
    print("Result rows:", len(result))
    print("Merge status:")
    print(result["_merge"].value_counts())
    return result

_ = merge_diagnostics(orders, customers, "customer_id")

## 5.14 Common Data Wrangling Interview Questions with Concise Answers

1. **What is the difference between `merge`, `join`, and `concat`?**  
   `merge` combines rows by key columns or indexes. `join` is a convenient mostly index-based join. `concat` stacks objects along rows or columns.

2. **What is the difference between an inner join and a left join?**  
   Inner join keeps only matching keys from both tables. Left join keeps all rows from the left table and adds matching right-side data when available.

3. **Why can a merge increase the number of rows?**  
   Duplicate keys on both sides create a many-to-many merge, producing multiple combinations for each repeated key.

4. **How do you check whether a merge worked correctly?**  
   Check row counts, duplicate keys, unmatched rows with `indicator=True`, expected relationship with `validate=`, and whether key metrics changed unexpectedly.

5. **What does `validate="many_to_one"` mean in `merge`?**  
   It means the left table may have repeated keys, but the right table should have each key at most once.

6. **What is an anti-join?**  
   An anti-join returns rows from one table that do not have a match in another table. In pandas, use `merge(..., indicator=True)` and filter `left_only`.

7. **When would you use `pd.concat`?**  
   Use it to append tables with the same schema, combine partitions, or place objects side by side along columns.

8. **What is a MultiIndex?**  
   A MultiIndex is an index with multiple levels, useful for grouped data, hierarchical labels, and reshape operations.

9. **What is the difference between `stack` and `unstack`?**  
   `stack` moves columns into the row index, making data longer. `unstack` moves an index level into columns, making data wider.

10. **What is the difference between `pivot` and `pivot_table`?**  
    `pivot` requires unique index-column pairs. `pivot_table` can handle duplicates by aggregating values.

11. **What does `melt` do?**  
    `melt` converts wide data into long format by turning columns into variable/value rows.

12. **What is tidy data?**  
    Tidy data means each variable is a column, each observation is a row, and each value is a cell.

## 5.15 Common Pitfalls

- Merging without checking whether keys are unique.
- Accidentally creating a many-to-many row explosion.
- Relying on pandas to infer join keys from overlapping column names.
- Using `inner` join and silently dropping unmatched rows.
- Forgetting `indicator=True` when debugging matches.
- Concatenating row partitions without `ignore_index=True` when the old index is meaningless.
- Using `pivot` when duplicate pairs require `pivot_table`.
- Reshaping data without knowing the current row grain.
- Keeping a complex MultiIndex when normal columns would be clearer.
- Failing to check totals before and after reshaping.

## 5.16 Mini Exercises

Try to answer before running the cells.

In [168]:
# Exercise 1: Why does this merge create more rows than the left table?
left = pd.DataFrame({"id": [1, 1, 2], "x": [10, 20, 30]})
right = pd.DataFrame({"id": [1, 1, 2], "y": [100, 200, 300]})

result = left.merge(right, on="id", how="left")
print(result)
print("Rows:", len(result))

   id   x    y
0   1  10  100
1   1  10  200
2   1  20  100
3   1  20  200
4   2  30  300
Rows: 5


In [169]:
# Exercise 2: Find users who have no purchases.
users = pd.DataFrame({"user_id": [1, 2, 3, 4]})
purchases = pd.DataFrame({"user_id": [2, 4], "amount": [50, 80]})

merged = users.merge(purchases, on="user_id", how="left", indicator=True)
no_purchase_users = merged.loc[merged["_merge"] == "left_only", ["user_id"]]
print(no_purchase_users)

   user_id
0        1
2        3


In [170]:
# Exercise 3: Convert wide monthly sales to long format.
wide_sales = pd.DataFrame({
    "store": ["A", "B"],
    "jan": [100, 80],
    "feb": [120, 90],
})

long_sales = wide_sales.melt(id_vars="store", var_name="month", value_name="sales")
print(long_sales)

  store month  sales
0     A   jan    100
1     B   jan     80
2     A   feb    120
3     B   feb     90


In [171]:
wide_sales

,store,jan,feb
0,A,100,120
1,B,80,90


## Part 6: Data Aggregation and Group Operations Interview Points

This section summarizes and extends Section 7 of `da_learn.ipynb`.

**Main interview theme:** `groupby` is not just syntax. It is a way to change the level of analysis: from raw rows to group-level summaries, group-level features, or custom per-group outputs.

## 6.1 Split-Apply-Combine Mental Model

`groupby` follows the split-apply-combine pattern:

1. **Split** rows into groups by one or more keys.
2. **Apply** a function to each group.
3. **Combine** results into a Series or DataFrame.

Interview framing:

- `agg`: one row per group, usually summary statistics.
- `transform`: same number of rows as original, usually group-level features.
- `apply`: flexible custom logic, but often slower and easier to misuse.

In [173]:
orders = pd.DataFrame({
    "order_id": range(1, 9),
    "customer_id": [101, 101, 102, 103, 103, 103, 104, 104],
    "region": ["East", "East", "East", "West", "West", "West", "East", "East"],
    "channel": ["web", "store", "web", "web", "store", "store", "web", "store"],
    "revenue": [120, 80, 75, 200, 50, 70, 30, 90],
    "discount": [10, 0, 5, 20, 0, 10, 0, 5],
})

display(orders)

,order_id,customer_id,region,channel,revenue,discount
0,1,101,East,web,120,10
1,2,101,East,store,80,0
2,3,102,East,web,75,5
3,4,103,West,web,200,20
4,5,103,West,store,50,0
5,6,103,West,store,70,10
6,7,104,East,web,30,0
7,8,104,East,store,90,5


In [174]:
region_summary = orders.groupby("region")["revenue"].sum()
print(region_summary)

region
East    395
West    320
Name: revenue, dtype: int64


**Interview answer template:**

> I use `groupby` when I need metrics at a different grain, such as customer-level spend from order-level rows or region-level revenue from transaction-level rows.

## 6.2 Single-Key and Multi-Key Grouping

Grouping by one key gives one level of groups. Grouping by multiple keys gives a combination of keys.

Use `as_index=False` when you want normal columns instead of a grouped index, especially for downstream merging or exporting.

In [175]:
summary = (
    orders
    .groupby(["region", "channel"], as_index=False)
    .agg(
        num_orders=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_revenue=("revenue", "mean"),
    )
)

print(summary)

  region channel  num_orders  total_revenue  avg_revenue
0   East   store           2            170         85.0
1   East     web           3            225         75.0
2   West   store           2            120         60.0
3   West     web           1            200        200.0


## 6.3 Named Aggregation: Clean Interview-Ready Style

Named aggregation gives readable output column names directly.

This is usually cleaner than aggregating first and renaming columns later.

In [176]:
customer_features = (
    orders
    .groupby("customer_id", as_index=False)
    .agg(
        total_spend=("revenue", "sum"),
        avg_order_value=("revenue", "mean"),
        num_orders=("order_id", "count"),
        max_discount=("discount", "max"),
    )
)

print(customer_features)

   customer_id  total_spend  avg_order_value  num_orders  max_discount
0          101          200       100.000000           2            10
1          102           75        75.000000           1             5
2          103          320       106.666667           3            20
3          104          120        60.000000           2             5


**Capability upgrade:** in interviews, prefer named aggregation because it shows you can produce clean feature tables, not just raw groupby outputs.

## 6.4 `size` vs `count`: Important Missing-Value Difference

- `size()` counts rows in each group, including missing values.
- `count()` counts non-missing values in each column.

This is a common interview trap.

In [177]:
tickets = pd.DataFrame({
    "team": ["A", "A", "A", "B", "B"],
    "resolved_hours": [2.5, np.nan, 4.0, np.nan, 1.0],
})

print("size counts rows:")
print(tickets.groupby("team").size())

print("\ncount counts non-missing values:")
print(tickets.groupby("team")["resolved_hours"].count())

size counts rows:
team
A    3
B    2
dtype: int64

count counts non-missing values:
team
A    2
B    1
Name: resolved_hours, dtype: int64


## 6.5 `agg` vs `transform` vs `apply`

| Method | Output shape | Best use |
|---|---|---|
| `agg` | usually fewer rows | group summaries |
| `transform` | same rows as original | group-level features aligned back to rows |
| `apply` | flexible | custom logic not covered by `agg`/`transform` |

Interview shortcut: if you need the result back on every original row, think `transform`.

In [179]:
orders["customer_avg_revenue"] = orders.groupby("customer_id")["revenue"].transform("mean")
orders["revenue_vs_customer_avg"] = orders["revenue"] - orders["customer_avg_revenue"]

display(orders[["order_id", "customer_id", "revenue", "customer_avg_revenue", "revenue_vs_customer_avg"]])

,order_id,customer_id,revenue,customer_avg_revenue,revenue_vs_customer_avg
0,1,101,120,100.000000,20.000000
1,2,101,80,100.000000,-20.000000
2,3,102,75,75.000000,0.000000
3,4,103,200,106.666667,93.333333
4,5,103,50,106.666667,-56.666667
5,6,103,70,106.666667,-36.666667
6,7,104,30,60.000000,-30.000000
7,8,104,90,60.000000,30.000000


In [180]:
# agg changes the grain: one row per customer.
print(orders.groupby("customer_id")["revenue"].agg(["sum", "mean", "count"]))

             sum        mean  count
customer_id                        
101          200  100.000000      2
102           75   75.000000      1
103          320  106.666667      3
104          120   60.000000      2


**Interview answer template:**

> `agg` reduces each group to summary rows. `transform` returns values aligned to the original rows. `apply` is more general but should not be the first choice if `agg` or `transform` can express the operation clearly.

## 6.6 Group-Specific Missing Value Imputation

Instead of filling missing values with one global value, you can fill using group-specific statistics.

This is often stronger for real data because different groups can have different distributions.

In [181]:
employees = pd.DataFrame({
    "department": ["Data", "Data", "Data", "Sales", "Sales", "Sales"],
    "salary": [100, np.nan, 120, 80, 90, np.nan],
})

employees["salary_filled"] = (
    employees
    .groupby("department")["salary"]
    .transform(lambda s: s.fillna(s.median()))
)

print(employees)

  department  salary  salary_filled
0       Data   100.0          100.0
1       Data     NaN          110.0
2       Data   120.0          120.0
3      Sales    80.0           80.0
4      Sales    90.0           90.0
5      Sales     NaN           85.0


**Interview caution:** for ML, compute imputation values on the training set only, then apply them to validation/test. Otherwise you leak information from future/unseen data.

## 6.7 Top-N Rows Per Group

A frequent interview task: find the top transaction, top product, top student, or top customer within each group.

Good options:

- `sort_values(...).groupby(...).head(n)`
- `groupby(...)[col].nlargest(n)`
- rank within group and filter

In [182]:
products = pd.DataFrame({
    "category": ["Book", "Book", "Book", "Game", "Game", "Game"],
    "product": ["B1", "B2", "B3", "G1", "G2", "G3"],
    "sales": [100, 180, 120, 90, 240, 150],
})

top2 = (
    products
    .sort_values(["category", "sales"], ascending=[True, False])
    .groupby("category", as_index=False)
    .head(2)
)

print(top2)

  category product  sales
1     Book      B2    180
2     Book      B3    120
4     Game      G2    240
5     Game      G3    150


In [183]:
products.sort_values(["category", "sales"], ascending=[True, False])

,category,product,sales
1,Book,B2,180
2,Book,B3,120
0,Book,B1,100
4,Game,G2,240
5,Game,G3,150
3,Game,G1,90


In [184]:
products["rank_in_category"] = products.groupby("category")["sales"].rank(
    method="dense",
    ascending=False,
)
print(products.sort_values(["category", "rank_in_category"]))

  category product  sales  rank_in_category
1     Book      B2    180               1.0
2     Book      B3    120               2.0
0     Book      B1    100               3.0
4     Game      G2    240               1.0
5     Game      G3    150               2.0
3     Game      G1     90               3.0


## 6.8 Filtering Groups

`groupby().filter()` keeps or removes entire groups based on group-level conditions.

Use cases:

- keep customers with at least 2 orders
- keep stores with enough observations
- remove groups with too much missing data

In [186]:
active_customers = orders.groupby("customer_id").filter(lambda g: len(g) >= 2)
print(active_customers[["order_id", "customer_id", "revenue"]])

   order_id  customer_id  revenue
0         1          101      120
1         2          101       80
3         4          103      200
4         5          103       50
5         6          103       70
6         7          104       30
7         8          104       90


In [187]:
orders

,order_id,customer_id,region,channel,revenue,discount,customer_avg_revenue,revenue_vs_customer_avg
0,1,101,East,web,120,10,100.000000,20.000000
1,2,101,East,store,80,0,100.000000,-20.000000
2,3,102,East,web,75,5,75.000000,0.000000
3,4,103,West,web,200,20,106.666667,93.333333
4,5,103,West,store,50,0,106.666667,-56.666667
5,6,103,West,store,70,10,106.666667,-36.666667
6,7,104,East,web,30,0,60.000000,-30.000000
7,8,104,East,store,90,5,60.000000,30.000000


## 6.9 Weighted Average by Group

Simple means can be misleading when observations have different importance.

Weighted averages are common in analytics interviews: average price weighted by volume, average score weighted by credits, average return weighted by portfolio weight.

In [188]:
campaigns = pd.DataFrame({
    "channel": ["Search", "Search", "Social", "Social"],
    "conversion_rate": [0.08, 0.12, 0.03, 0.05],
    "impressions": [1000, 3000, 5000, 1000],
})

def weighted_average(group, value_col, weight_col):
    return np.average(group[value_col], weights=group[weight_col])

weighted_cr = campaigns.groupby("channel").apply(
    weighted_average,
    value_col="conversion_rate",
    weight_col="impressions",
)

print(weighted_cr)

channel
Search    0.110000
Social    0.033333
dtype: float64


/var/folders/_5/ztyxm4j95854xqs2k_x83m1c0000gn/T/ipykernel_6259/1091742081.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_cr = campaigns.groupby("channel").apply(


**Interview explanation:**

> A regular average treats each row equally. A weighted average gives more influence to rows with larger weights, such as more impressions, more volume, or more credits.

## 6.10 Bucket Analysis with `cut` / `qcut` + `groupby`

Bucket analysis helps answer questions like:

- Do higher spend users churn less?
- Does conversion rate vary by age band?
- What is the average target by prediction score bucket?

`pd.cut` uses fixed bins. `pd.qcut` uses quantile bins with roughly equal counts.

In [189]:
customers = pd.DataFrame({
    "customer_id": range(1, 11),
    "spend": [20, 35, 50, 80, 120, 160, 220, 300, 450, 700],
    "churned": [1, 1, 0, 0, 0, 1, 0, 0, 1, 0],
})

customers["spend_bucket"] = pd.qcut(customers["spend"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

bucket_summary = customers.groupby("spend_bucket", observed=True).agg(
    customers=("customer_id", "count"),
    avg_spend=("spend", "mean"),
    churn_rate=("churned", "mean"),
)
print(bucket_summary)

              customers   avg_spend  churn_rate
spend_bucket                                   
Q1                    3   35.000000    0.666667
Q2                    2  100.000000    0.000000
Q3                    2  190.000000    0.500000
Q4                    3  483.333333    0.333333


## 6.11 Pivot Tables

A pivot table is a compact way to aggregate data across two or more dimensions.

It is conceptually `groupby + aggregation + reshape`.

Useful arguments:

- `index=` rows
- `columns=` columns
- `values=` metric
- `aggfunc=` aggregation function
- `margins=True` totals

In [190]:
sales = pd.DataFrame({
    "region": ["East", "East", "East", "West", "West", "West"],
    "channel": ["web", "store", "web", "web", "store", "store"],
    "revenue": [100, 80, 120, 200, 50, 70],
})

pivot = sales.pivot_table(
    index="region",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    margins=True,
    fill_value=0,
)
print(pivot)

channel  store  web  All
region                  
East        80  220  300
West       120  200  320
All        200  420  620


## 6.12 Crosstab for Counts and Proportions

`pd.crosstab` is convenient for frequency tables.

Use cases:

- survey counts
- confusion-matrix-like summaries
- category by category distributions

With `normalize=`, it can return proportions.

In [191]:
survey = pd.DataFrame({
    "country": ["US", "US", "US", "JP", "JP", "JP", "JP"],
    "device": ["iOS", "Android", "iOS", "Android", "Android", "iOS", "iOS"],
})

print(pd.crosstab(survey["country"], survey["device"], margins=True))
print("\nRow proportions:")
print(pd.crosstab(survey["country"], survey["device"], normalize="index"))

device   Android  iOS  All
country                   
JP             2    2    4
US             1    2    3
All            3    4    7

Row proportions:
device    Android       iOS
country                    
JP       0.500000  0.500000
US       0.333333  0.666667


## 6.13 Feature Engineering with GroupBy

For data analyst / DS / MLE interviews, groupby often appears as feature engineering.

Examples:

- customer lifetime value
- number of orders per customer
- user's average spend
- product-level conversion rate
- difference from group average
- within-group rank

Strong habit: after creating group-level features, check whether they leak future information.

In [193]:
events = pd.DataFrame({
    "user_id": [1, 1, 1, 2, 2, 3],
    "event_time": pd.to_datetime([
        "2026-01-01", "2026-01-03", "2026-01-05",
        "2026-01-02", "2026-01-04", "2026-01-01",
    ]),
    "purchase_amount": [0, 20, 30, 0, 50, 10],
})

events = events.sort_values(["user_id", "event_time"])
events["prior_user_spend"] = (
    events
    .groupby("user_id")["purchase_amount"]
    .apply(lambda x: x.cumsum().shift(fill_value=0))
    .reset_index(level=0, drop=True) # 移除 groupby 自动生成的额外索引
)

display(events)

,user_id,event_time,purchase_amount,prior_user_spend
0,1,2026-01-01,0,0
1,1,2026-01-03,20,0
2,1,2026-01-05,30,20
3,2,2026-01-02,0,0
4,2,2026-01-04,50,0
5,3,2026-01-01,10,0


**Leakage note:** the example uses prior cumulative spend. For prediction tasks, this is safer than using each user's total spend computed from all future rows.

## 6.14 Common GroupBy Interview Questions with Concise Answers

1. **What is split-apply-combine?**  
   It means split data into groups, apply a function to each group, then combine the results.

2. **What is the difference between `agg`, `transform`, and `apply`?**  
   `agg` returns group-level summaries, `transform` returns values aligned to the original rows, and `apply` handles flexible custom per-group logic.

3. **When would you use `transform`?**  
   Use it when you need a group-level statistic repeated back on each original row, such as customer average spend on every order row.

4. **What is the difference between `size` and `count`?**  
   `size` counts all rows in each group. `count` counts non-missing values.

5. **How do you aggregate multiple columns with different functions?**  
   Use named aggregation inside `.agg`, such as `total=("revenue", "sum")` and `avg=("revenue", "mean")`.

6. **How do you get the top N rows per group?**  
   Sort within groups and use `groupby(...).head(n)`, or rank within group and filter.

7. **How do you fill missing values using group-specific values?**  
   Use `groupby(...).transform(lambda s: s.fillna(s.median()))` or a similar group-specific statistic.

8. **Why can `apply` be risky?**  
   It is flexible but can be slower, harder to reason about, and may produce awkward indexes. Prefer `agg` or `transform` when possible.

9. **What is a pivot table?**  
   A pivot table summarizes values across row and column categories, usually by aggregation. It is like `groupby` plus reshape.

10. **What is `crosstab` used for?**  
    It creates frequency tables between categorical variables and can also show proportions.

11. **What is a weighted average and why use it?**  
    A weighted average gives more influence to rows with larger weights, useful when observations have different importance.

12. **What is data leakage in groupby feature engineering?**  
    Leakage happens when group features use information that would not be available at prediction time, such as future purchases.

## 6.15 Common Pitfalls

- Using `count()` when you meant to count all rows; use `size()` instead.
- Forgetting `as_index=False` when you want normal columns after aggregation.
- Using `apply` when named `agg` or `transform` is clearer.
- Creating group features from future data and causing leakage.
- Comparing group means without checking group sizes.
- Forgetting that `groupby` drops missing group keys by default in many cases.
- Producing a MultiIndex and not resetting it when downstream code expects flat columns.
- Using simple averages when weighted averages are appropriate.
- Treating pivot tables as raw data instead of aggregated summaries.
- Ignoring small groups whose metrics may be noisy.

## 6.16 Mini Exercises

Try to answer before running the cells.

In [194]:
# Exercise 1: Create customer-level features.
orders = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 2, 3],
    "amount": [20, 30, 10, 40, 50, 15],
})

features = orders.groupby("customer_id", as_index=False).agg(
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
    num_orders=("amount", "size"),
)
print(features)

   customer_id  total_amount  avg_amount  num_orders
0            1            50   25.000000           2
1            2           100   33.333333           3
2            3            15   15.000000           1


In [195]:
# Exercise 2: Add each row's difference from its group mean.
orders["customer_avg"] = orders.groupby("customer_id")["amount"].transform("mean")
orders["diff_from_avg"] = orders["amount"] - orders["customer_avg"]
print(orders)

   customer_id  amount  customer_avg  diff_from_avg
0            1      20     25.000000      -5.000000
1            1      30     25.000000       5.000000
2            2      10     33.333333     -23.333333
3            2      40     33.333333       6.666667
4            2      50     33.333333      16.666667
5            3      15     15.000000       0.000000


In [196]:
# Exercise 3: Top 1 product by sales within each category.
products = pd.DataFrame({
    "category": ["A", "A", "B", "B"],
    "product": ["p1", "p2", "p3", "p4"],
    "sales": [100, 150, 80, 120],
})

top_product = products.sort_values("sales", ascending=False).groupby("category").head(1)
print(top_product.sort_values("category"))

  category product  sales
1        A      p2    150
3        B      p4    120


## Part 7: Time Series Interview Points

This section is based on Wes McKinney's *Python for Data Analysis*, Chapter 11: Time Series.

**Main interview theme:** time series analysis is about respecting time order. The most common mistakes are parsing dates incorrectly, mixing time zones, resampling with the wrong aggregation, and creating features that leak future information.

## 7.1 Core Time Series Objects

Important objects:

- `datetime`: Python standard library object for one timestamp.
- `Timedelta`: difference between two times.
- `pd.Timestamp`: pandas timestamp scalar.
- `DatetimeIndex`: pandas index made of timestamps.
- `Period`: a span of time such as one month or one quarter.
- `NaT`: pandas missing value for datetime-like data.

Interview point: `Timestamp` represents a point in time; `Period` represents a time span.

In [198]:
stamp = pd.Timestamp("2026-05-18 09:30:00")
print(stamp)
print("year:", stamp.year)
print("weekday:", stamp.day_name())

period = pd.Period("2026-05", freq="M")
print("period:", period)
print("period start:", period.start_time)
print("period end:", period.end_time)

2026-05-18 09:30:00
year: 2026
weekday: Monday
period: 2026-05
period start: 2026-05-01 00:00:00
period end: 2026-05-31 23:59:59.999999999


## 7.2 Parsing Dates Safely

Use `pd.to_datetime` for vectorized parsing.

Practical tips:

- Use `format=` when the date format is known.
- Use `errors="coerce"` to turn bad dates into `NaT` instead of crashing.
- Check how many `NaT` values were created.
- Be careful with ambiguous formats like `01/02/2026`.

In [199]:
raw_dates = pd.Series(["2026-01-05", "2026-02-10", "bad-date", None])
parsed_dates = pd.to_datetime(raw_dates, errors="coerce")

print(parsed_dates)
print("Missing datetime values:", parsed_dates.isna().sum())

0   2026-01-05
1   2026-02-10
2          NaT
3          NaT
dtype: datetime64[ns]
Missing datetime values: 2


In [200]:
# Known format: faster and less ambiguous.
us_dates = pd.Series(["05/18/2026", "05/19/2026"])
parsed_us_dates = pd.to_datetime(us_dates, format="%m/%d/%Y")
print(parsed_us_dates)

0   2026-05-18
1   2026-05-19
dtype: datetime64[ns]


**Interview answer template:**

> I parse dates with `pd.to_datetime`, specify `format` when possible, use `errors="coerce"` for dirty data, and then audit `NaT` values before analysis.

## 7.3 `DatetimeIndex` and Time-Based Selection

A `DatetimeIndex` allows clean time slicing.

Useful patterns:

- select one exact date
- partial string slicing by year or month
- range slicing
- sort by time before slicing or modeling

In [202]:
daily_sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=10, freq="D"),
    "sales": [100, 120, 90, 130, 160, 150, 170, 180, 140, 200],
})

sales_ts = daily_sales.set_index("date").sort_index()
print(sales_ts)

print("\nFirst week:")
print(sales_ts.loc["2026-01-01":"2026-01-07"])

print("\nAll January rows:")
print(sales_ts.loc["2026-01"])

            sales
date             
2026-01-01    100
2026-01-02    120
2026-01-03     90
2026-01-04    130
2026-01-05    160
2026-01-06    150
2026-01-07    170
2026-01-08    180
2026-01-09    140
2026-01-10    200

First week:
            sales
date             
2026-01-01    100
2026-01-02    120
2026-01-03     90
2026-01-04    130
2026-01-05    160
2026-01-06    150
2026-01-07    170

All January rows:
            sales
date             
2026-01-01    100
2026-01-02    120
2026-01-03     90
2026-01-04    130
2026-01-05    160
2026-01-06    150
2026-01-07    170
2026-01-08    180
2026-01-09    140
2026-01-10    200


## 7.4 Duplicate Timestamps

Time indexes are not always unique. Multiple events can happen at the same timestamp.

Common options:

- keep duplicate timestamps for event-level analysis
- aggregate duplicates using `groupby(level=0)`
- add another key, such as user ID or event ID

Interview point: always check whether your time index is unique before assuming one row per timestamp.

In [203]:
events = pd.DataFrame({
    "time": pd.to_datetime(["2026-01-01 09:00", "2026-01-01 09:00", "2026-01-01 10:00"]),
    "clicks": [1, 2, 3],
}).set_index("time")

print(events)
print("Is index unique?", events.index.is_unique)

hourly_clicks = events.groupby(level=0).sum()
print("\nAggregated duplicate timestamps:")
print(hourly_clicks)

                     clicks
time                       
2026-01-01 09:00:00       1
2026-01-01 09:00:00       2
2026-01-01 10:00:00       3
Is index unique? False

Aggregated duplicate timestamps:
                     clicks
time                       
2026-01-01 09:00:00       3
2026-01-01 10:00:00       3


## 7.5 Date Ranges and Frequencies

`pd.date_range` creates regular time indexes.

Common frequency aliases:

- `D`: calendar day
- `B`: business day
- `W`: weekly
- `M`: month end
- `MS`: month start
- `H`: hourly
- `min`: minute

Interview point: understand the difference between calendar days and business days.

In [204]:
calendar_days = pd.date_range("2026-01-01", periods=5, freq="D")
business_days = pd.date_range("2026-01-01", periods=5, freq="B")

print("Calendar days:")
print(calendar_days)
print("\nBusiness days:")
print(business_days)

Calendar days:
DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04',
               '2026-01-05'],
              dtype='datetime64[ns]', freq='D')

Business days:
DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-05', '2026-01-06',
               '2026-01-07'],
              dtype='datetime64[ns]', freq='B')


## 7.6 Shifting, Lag Features, and Percent Change

`shift` moves values relative to the index. It is commonly used for lag features.

Common uses:

- yesterday's value
- previous week's sales
- return / growth rate
- change from previous period

Interview point: lag features should use only past information. This is a major leakage topic.

In [206]:
traffic = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=6, freq="D"),
    "visits": [100, 120, 90, 150, 180, 170],
}).set_index("date")

traffic["visits_lag_1"] = traffic["visits"].shift(1)
traffic["daily_change"] = traffic["visits"] - traffic["visits_lag_1"]
traffic["pct_change"] = traffic["visits"].pct_change()

display(traffic)

,visits,visits_lag_1,daily_change,pct_change
date,,,,
2026-01-01,100,NaN,NaN,NaN
2026-01-02,120,100.0,20.0,0.200000
2026-01-03,90,120.0,-30.0,-0.250000
2026-01-04,150,90.0,60.0,0.666667
2026-01-05,180,150.0,30.0,0.200000
2026-01-06,170,180.0,-10.0,-0.055556


**Leakage-safe sentence:**

> For predictive modeling, I create lagged or rolling features from past values only. I avoid using future rows when computing features for a given timestamp.

## 7.7 Time Zones: `tz_localize` vs `tz_convert`

Time zone handling is a common source of subtle bugs.

- `tz_localize`: assign a time zone to naive timestamps.
- `tz_convert`: convert time-zone-aware timestamps to another zone.

Interview point: localize first, then convert. Do not confuse them.

In [207]:
naive_times = pd.date_range("2026-05-18 09:00", periods=3, freq="h")
sg_times = naive_times.tz_localize("Asia/Singapore")
ny_times = sg_times.tz_convert("America/New_York")

print("Singapore times:")
print(sg_times)
print("\nNew York times:")
print(ny_times)

Singapore times:
DatetimeIndex(['2026-05-18 09:00:00+08:00', '2026-05-18 10:00:00+08:00',
               '2026-05-18 11:00:00+08:00'],
              dtype='datetime64[ns, Asia/Singapore]', freq=None)

New York times:
DatetimeIndex(['2026-05-17 21:00:00-04:00', '2026-05-17 22:00:00-04:00',
               '2026-05-17 23:00:00-04:00'],
              dtype='datetime64[ns, America/New_York]', freq=None)


## 7.8 Resampling: Change the Time Frequency

`resample` is like time-based `groupby`.

Common patterns:

- downsample minute data to hourly/daily summaries
- upsample daily data to hourly/daily slots
- fill missing periods after upsampling

Choose aggregation based on meaning:

- sales/revenue/counts: usually `sum`
- temperature/price level: often `mean` or last value
- inventory/account balance: often last value

In [208]:
minute_data = pd.DataFrame({
    "time": pd.date_range("2026-01-01 09:00", periods=12, freq="5min"),
    "orders": [1, 0, 2, 1, 3, 0, 2, 2, 1, 0, 1, 4],
}).set_index("time")

hourly_orders = minute_data.resample("h").sum()
print(minute_data.head())
print("\nHourly orders:")
print(hourly_orders)

                     orders
time                       
2026-01-01 09:00:00       1
2026-01-01 09:05:00       0
2026-01-01 09:10:00       2
2026-01-01 09:15:00       1
2026-01-01 09:20:00       3

Hourly orders:
                     orders
time                       
2026-01-01 09:00:00      17


In [209]:
daily_inventory = pd.DataFrame({
    "date": pd.to_datetime(["2026-01-01", "2026-01-03", "2026-01-06"]),
    "inventory": [100, 90, 120],
}).set_index("date")

filled_daily_inventory = daily_inventory.resample("D").ffill()
print(filled_daily_inventory)

            inventory
date                 
2026-01-01        100
2026-01-02        100
2026-01-03         90
2026-01-04         90
2026-01-05         90
2026-01-06        120


## 7.9 `asfreq` vs `resample`

- `asfreq` changes the index frequency without aggregation.
- `resample` groups timestamps into time bins and then aggregates or fills.

Interview shortcut:

- Need a new regular index? Think `asfreq`.
- Need time-window summaries? Think `resample`.

In [210]:
sparse = pd.Series(
    [10, 20, 30],
    index=pd.to_datetime(["2026-01-01", "2026-01-03", "2026-01-06"]),
)

print("asfreq daily:")
print(sparse.asfreq("D"))

print("\nresample daily with forward fill:")
print(sparse.resample("D").ffill())

asfreq daily:
2026-01-01    10.0
2026-01-02     NaN
2026-01-03    20.0
2026-01-04     NaN
2026-01-05     NaN
2026-01-06    30.0
Freq: D, dtype: float64

resample daily with forward fill:
2026-01-01    10
2026-01-02    10
2026-01-03    20
2026-01-04    20
2026-01-05    20
2026-01-06    30
Freq: D, dtype: int64


## 7.10 Grouped Time Resampling

For multiple time series in one table, combine normal groups with time bins.

Use `pd.Grouper(key="time", freq="...")` when the timestamp is a column.

This is very useful for event logs, user activity, IoT data, and transaction data.

In [211]:
logs = pd.DataFrame({
    "time": pd.date_range("2026-01-01 09:00", periods=8, freq="15min"),
    "app": ["A", "A", "B", "A", "B", "B", "A", "B"],
    "requests": [10, 20, 5, 15, 10, 20, 30, 25],
})

app_hourly = (
    logs
    .groupby(["app", pd.Grouper(key="time", freq="h")])
    .agg(total_requests=("requests", "sum"))
    .reset_index()
)

print(app_hourly)

  app                time  total_requests
0   A 2026-01-01 09:00:00              45
1   A 2026-01-01 10:00:00              30
2   B 2026-01-01 09:00:00               5
3   B 2026-01-01 10:00:00              55


## 7.11 Rolling, Expanding, and EWM Windows

Window functions create features or summaries over nearby time points.

- `rolling(window=...)`: fixed-size moving window
- `expanding()`: from the start through current row
- `ewm(...)`: exponentially weighted, more weight on recent observations

Interview point: rolling features are widely used for trend and volatility, but they must be aligned carefully to avoid future leakage.

In [212]:
price = pd.Series(
    [100, 102, 101, 105, 107, 106, 110],
    index=pd.date_range("2026-01-01", periods=7, freq="D"),
    name="price",
)

features = pd.DataFrame({
    "price": price,
    "rolling_3_mean": price.rolling(window=3, min_periods=2).mean(),
    "expanding_mean": price.expanding(min_periods=2).mean(),
    "ewm_mean": price.ewm(span=3, adjust=False).mean(),
})

print(features)

            price  rolling_3_mean  expanding_mean  ewm_mean
2026-01-01    100             NaN             NaN    100.00
2026-01-02    102      101.000000      101.000000    101.00
2026-01-03    101      101.000000      101.000000    101.00
2026-01-04    105      102.666667      102.000000    103.00
2026-01-05    107      104.333333      103.000000    105.00
2026-01-06    106      106.000000      103.500000    105.50
2026-01-07    110      107.666667      104.428571    107.75


| 特征 | 窗口大小 | 老数据的权重 | 典型应用场景 |
|---|---|---|---|
| rolling | 固定（本例为 3 天） | 只要移出窗口，权重瞬间归 0 | 短期趋势、技术指标（如 MA5） |
| expanding | 持续变大（无限延伸） | 所有人平分权重，地位均等 | 计算历史至今的累计指标（如累计平均成本） |
| ewm | 无限大（包含所有历史） | 时间越久远，权重以指数级无限衰减 | 经典的趋势跟踪（如 EMA 均线、波动率估计） |

In [213]:
returns = price.pct_change()
volatility = returns.rolling(window=3, min_periods=2).std()

print(pd.DataFrame({"return": returns, "rolling_volatility": volatility}))

              return  rolling_volatility
2026-01-01       NaN                 NaN
2026-01-02  0.020000                 NaN
2026-01-03 -0.009804            0.021075
2026-01-04  0.039604            0.024879
2026-01-05  0.019048            0.024820
2026-01-06 -0.009346            0.024579
2026-01-07  0.037736            0.023707


## 7.12 Time-Aware Feature Engineering

Common time features:

- hour of day
- day of week
- weekend flag
- month
- quarter
- lag features
- rolling summaries
- time since previous event

Interview point: for modeling, split train/test by time, not random shuffle, when predicting future behavior.

In [215]:
transactions = pd.DataFrame({
    "time": pd.to_datetime([
        "2026-01-01 09:00", "2026-01-01 10:30", "2026-01-03 12:00",
        "2026-01-05 09:15", "2026-01-06 16:00",
    ]),
    "amount": [20, 35, 15, 50, 40],
}).sort_values("time")

transactions["hour"] = transactions["time"].dt.hour
transactions["day_of_week"] = transactions["time"].dt.day_name()
transactions["is_weekend"] = transactions["time"].dt.dayofweek >= 5
transactions["time_since_previous"] = transactions["time"].diff()

print(transactions)

                 time  amount  hour day_of_week  is_weekend  \
0 2026-01-01 09:00:00      20     9    Thursday       False   
1 2026-01-01 10:30:00      35    10    Thursday       False   
2 2026-01-03 12:00:00      15    12    Saturday        True   
3 2026-01-05 09:15:00      50     9      Monday       False   
4 2026-01-06 16:00:00      40    16     Tuesday       False   

  time_since_previous  
0                 NaT  
1     0 days 01:30:00  
2     2 days 01:30:00  
3     1 days 21:15:00  
4     1 days 06:45:00  


In [216]:
# Simple time-based train/test split.
cutoff = pd.Timestamp("2026-01-05")
train = transactions[transactions["time"] < cutoff]
test = transactions[transactions["time"] >= cutoff]

print("Train:")
print(train)
print("\nTest:")
print(test)

Train:
                 time  amount  hour day_of_week  is_weekend  \
0 2026-01-01 09:00:00      20     9    Thursday       False   
1 2026-01-01 10:30:00      35    10    Thursday       False   
2 2026-01-03 12:00:00      15    12    Saturday        True   

  time_since_previous  
0                 NaT  
1     0 days 01:30:00  
2     2 days 01:30:00  

Test:
                 time  amount  hour day_of_week  is_weekend  \
3 2026-01-05 09:15:00      50     9      Monday       False   
4 2026-01-06 16:00:00      40    16     Tuesday       False   

  time_since_previous  
3     1 days 21:15:00  
4     1 days 06:45:00  


## 7.13 Common Time Series Interview Questions with Concise Answers

1. **What is a `DatetimeIndex`?**  
   A pandas index made of timestamps, enabling time-based slicing, alignment, resampling, and rolling operations.

2. **What is `NaT`?**  
   `NaT` means Not a Time. It is pandas' missing value for datetime-like data.

3. **How do you safely parse date strings?**  
   Use `pd.to_datetime`, specify `format` when known, use `errors="coerce"` for dirty data, and inspect resulting `NaT` values.

4. **What is the difference between `Timestamp` and `Period`?**  
   A `Timestamp` is a point in time. A `Period` is a span of time, such as a month or quarter.

5. **What is the difference between `tz_localize` and `tz_convert`?**  
   `tz_localize` assigns a timezone to naive timestamps. `tz_convert` converts timezone-aware timestamps to another timezone.

6. **What does `shift` do in time series?**  
   It moves values forward or backward relative to the index and is commonly used to create lag features.

7. **What is resampling?**  
   Resampling changes the time frequency, such as converting minute data to hourly data or daily data to monthly data.

8. **What is the difference between downsampling and upsampling?**  
   Downsampling aggregates high-frequency data to lower frequency. Upsampling expands lower-frequency data to higher frequency and often creates missing values to fill.

9. **What is the difference between `asfreq` and `resample`?**  
   `asfreq` changes the frequency without aggregation. `resample` groups data into time bins and then aggregates or fills.

10. **What is a rolling window?**  
    A rolling window computes statistics over a fixed number of recent observations or a fixed time span.

11. **What is the difference between rolling and expanding windows?**  
    Rolling uses a fixed-size moving window. Expanding uses all observations from the start up to the current row.

12. **Why is time series leakage dangerous?**  
    It happens when features use future information that would not be available at prediction time, causing overly optimistic model performance.

## 7.14 Common Pitfalls

- Parsing dates without checking failed parses or ambiguous formats.
- Treating strings as dates instead of converting to datetime dtype.
- Forgetting to sort by time before time slicing, rolling, or modeling.
- Using random train/test split for future prediction tasks.
- Confusing `tz_localize` with `tz_convert`.
- Resampling with the wrong aggregation, such as averaging sales instead of summing sales.
- Forgetting that upsampling creates missing values.
- Creating lag or rolling features that accidentally use future data.
- Ignoring duplicate timestamps.
- Comparing time series with different time zones or frequencies without alignment.

## 7.15 Mini Exercises

Try to answer before running the cells.

In [217]:
# Exercise 1: Parse dirty dates and count failures.
dirty_dates = pd.Series(["2026-01-01", "not a date", "2026-02-01", ""])
parsed = pd.to_datetime(dirty_dates, errors="coerce")
print(parsed)
print("Failed parses:", parsed.isna().sum())

0   2026-01-01
1          NaT
2   2026-02-01
3          NaT
dtype: datetime64[ns]
Failed parses: 2


In [218]:
# Exercise 2: Create lag and rolling mean features.
visits = pd.Series(
    [100, 120, 90, 150, 130],
    index=pd.date_range("2026-01-01", periods=5, freq="D"),
)

features = pd.DataFrame({
    "visits": visits,
    "lag_1": visits.shift(1),
    "rolling_3_mean": visits.rolling(3, min_periods=1).mean(),
})
print(features)

            visits  lag_1  rolling_3_mean
2026-01-01     100    NaN      100.000000
2026-01-02     120  100.0      110.000000
2026-01-03      90  120.0      103.333333
2026-01-04     150   90.0      120.000000
2026-01-05     130  150.0      123.333333


In [219]:
# Exercise 3: Resample event-level revenue to daily revenue.
events = pd.DataFrame({
    "time": pd.to_datetime([
        "2026-01-01 09:00", "2026-01-01 12:00", "2026-01-02 10:00"
    ]),
    "revenue": [20, 30, 40],
}).set_index("time")

daily_revenue = events.resample("D")["revenue"].sum()
print(daily_revenue)

time
2026-01-01    50
2026-01-02    40
Freq: D, Name: revenue, dtype: int64
